<a href="https://colab.research.google.com/github/Mridul33/capstone-project/blob/main/Stage_1-3.1_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 1: BioRED Dataset Inspection, Preprocessing and Validation

This stage loads the BioRED train, development and test splits, inspects the
dataset structure, reconstructs title-and-abstract text using the original
passage offsets, and validates entity offsets and relation references before
model-ready data preparation.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

In [2]:
!wget https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/BIORED.zip
!unzip -q BIORED.zip -d BioRED_data

--2026-08-16 08:16:23--  https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/BIORED.zip
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.31, 130.14.250.7, 2607:f220:41e:250::11, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2191684 (2.1M) [application/zip]
Saving to: ‘BIORED.zip’

BIORED.zip          100%[===================>]   2.09M  5.02MB/s    in 0.4s    

2026-08-16 08:16:24 (5.02 MB/s) - ‘BIORED.zip’ saved [2191684/2191684]



In [3]:
!find BioRED_data -maxdepth 3 -type f | head -50

BioRED_data/BioRED/Test.PubTator
BioRED_data/BioRED/Train.BioC.XML
BioRED_data/BioRED/Test.BioC.JSON
BioRED_data/BioRED/Dev.BioC.XML
BioRED_data/BioRED/Train.BioC.JSON
BioRED_data/BioRED/Dev.BioC.JSON
BioRED_data/BioRED/Train.PubTator
BioRED_data/BioRED/Test.BioC.XML
BioRED_data/BioRED/Dev.PubTator


In [4]:
data_dir = Path("/content/BioRED_data/BioRED")

train_file = data_dir / "Train.BioC.JSON"
dev_file = data_dir / "Dev.BioC.JSON"
test_file = data_dir / "Test.BioC.JSON"


def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)


train_data = load_json(train_file)
dev_data = load_json(dev_file)
test_data = load_json(test_file)

print("BioRED datasets loaded successfully.")

BioRED datasets loaded successfully.


In [5]:
print("Training documents:", len(train_data["documents"]))
print("Development documents:", len(dev_data["documents"]))
print("Test documents:", len(test_data["documents"]))
print("Train keys:", train_data.keys())
print("First training document keys:", train_data["documents"][0].keys())
print("Dev keys:", dev_data.keys())
print("Test keys:", test_data.keys())

Training documents: 400
Development documents: 100
Test documents: 100
Train keys: dict_keys(['source', 'date', 'key', 'documents'])
First training document keys: dict_keys(['id', 'passages', 'relations'])
Dev keys: dict_keys(['source', 'date', 'key', 'documents'])
Test keys: dict_keys(['source', 'date', 'key', 'documents'])


In [6]:
first_document = train_data["documents"][0]

print(first_document.keys())
print("Document ID:", first_document["id"])
print("Number of passages:", len(first_document["passages"]))

dict_keys(['id', 'passages', 'relations'])
Document ID: 10491763
Number of passages: 2


In [7]:
from pprint import pprint

print("Document ID:", first_document["id"])
print("Number of passages:", len(first_document["passages"]))

for i, passage in enumerate(first_document["passages"]):
    print(f"\nPassage {i}")
    pprint(passage)

Document ID: 10491763
Number of passages: 2

Passage 0
{'annotations': [{'id': '0',
                  'infons': {'identifier': '3175', 'type': 'GeneOrGeneProduct'},
                  'locations': [{'length': 27, 'offset': 0}],
                  'text': 'Hepatocyte nuclear factor-6'},
                 {'id': '1',
                  'infons': {'identifier': 'D003924',
                             'type': 'DiseaseOrPhenotypicFeature'},
                  'locations': [{'length': 16, 'offset': 74}],
                  'text': 'type II diabetes'},
                 {'id': '2',
                  'infons': {'identifier': '3630', 'type': 'GeneOrGeneProduct'},
                  'locations': [{'length': 7, 'offset': 140}],
                  'text': 'insulin'}],
 'offset': 0,
 'text': 'Hepatocyte nuclear factor-6: associations between genetic '
         'variability and type II diabetes and between genetic variability and '
         'estimates of insulin secretion.'}

Passage 1
{'annotations': [{'id'

In [8]:
print("Document ID:", first_document["id"])

for i, passage in enumerate(first_document["passages"]):

    if i == 0:
        passage_type = "title"
    elif i == 1:
        passage_type = "abstract"
    else:
        passage_type = f"passage_{i}"

    # Store the label inside the passage dictionary
    passage["passage_type"] = passage_type

    print("\nPassage number:", i)
    print("Type:", passage["passage_type"])
    print("Offset:", passage.get("offset"))
    print("Text:", passage.get("text"))
    print("Annotations:", len(passage.get("annotations", [])))
    print("-" * 50)

Document ID: 10491763

Passage number: 0
Type: title
Offset: 0
Text: Hepatocyte nuclear factor-6: associations between genetic variability and type II diabetes and between genetic variability and estimates of insulin secretion.
Annotations: 3
--------------------------------------------------

Passage number: 1
Type: abstract
Offset: 159
Text: The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mutations in the MODY1, MODY3 and MODY4 genes and in 54 patients with late-onset Type II diabetes by combined singl

In [9]:
title_text = first_document["passages"][0]["text"]
abstract_offset = first_document["passages"][1]["offset"]

print("Title length:", len(title_text))
print("Abstract offset:", abstract_offset)
print("Gap between title and abstract:", abstract_offset - len(title_text))

Title length: 158
Abstract offset: 159
Gap between title and abstract: 1


In [10]:
print("Document ID:", first_document["id"])

passages = first_document["passages"]

document_length = max(
    passage.get("offset", 0) + len(passage.get("text", ""))
    for passage in passages
)

full_text_chars = [" "] * document_length

for passage in passages:
    passage_text = passage.get("text", "")
    passage_offset = passage.get("offset", 0)

    print("\nPassage type:", passage["passage_type"])
    print("Offset:", passage_offset)
    print("Text:", passage_text)

    start = passage_offset
    end = start + len(passage_text)

    full_text_chars[start:end] = list(passage_text)

full_text = "".join(full_text_chars)

print("\nCombined document text:\n")
print(full_text)

Document ID: 10491763

Passage type: title
Offset: 0
Text: Hepatocyte nuclear factor-6: associations between genetic variability and type II diabetes and between genetic variability and estimates of insulin secretion.

Passage type: abstract
Offset: 159
Text: The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mutations in the MODY1, MODY3 and MODY4 genes and in 54 patients with late-onset Type II diabetes by combined single strand conformational polymorphism-heteroduplex analysis followed by direct sequenci

In [11]:
import pandas as pd

entity_rows = []

for passage in first_document["passages"]:
    for annotation in passage.get("annotations", []):
        location = annotation["locations"][0]

        start = location["offset"]
        length = location["length"]
        end = start + length

        annotated_text = annotation.get("text", "")
        text_from_offset = full_text[start:end]

        entity_rows.append({
            "document_id": first_document["id"],
            "entity_id": annotation["id"],
            "identifier": annotation.get("infons", {}).get("identifier"),
            "entity_type": annotation.get("infons", {}).get("type"),
            "entity_text": annotated_text,
            "offset": start,
            "length": length,
            "end_offset": end,
            "passage_type": passage["passage_type"],
            "text_from_offset": text_from_offset,
            "offset_match": annotated_text == text_from_offset
        })

entity_df = pd.DataFrame(entity_rows)

display(entity_df)

,document_id,entity_id,identifier,entity_type,entity_text,offset,length,end_offset,passage_type,text_from_offset,offset_match
0,10491763,0,3175,GeneOrGeneProduct,Hepatocyte nuclear factor-6,0,27,27,title,Hepatocyte nuclear factor-6,True
1,10491763,1,D003924,DiseaseOrPhenotypicFeature,type II diabetes,74,16,90,title,type II diabetes,True
2,10491763,2,3630,GeneOrGeneProduct,insulin,140,7,147,title,insulin,True
3,10491763,3,3175,GeneOrGeneProduct,hepatocyte nuclear factor (HNF)-6,184,33,217,abstract,hepatocyte nuclear factor (HNF)-6,True
4,10491763,4,D003924,DiseaseOrPhenotypicFeature,maturity-onset diabetes,292,23,315,abstract,maturity-onset diabetes,True
5,10491763,5,3175,GeneOrGeneProduct,HNF-6,389,5,394,abstract,HNF-6,True
6,10491763,6,D003924,DiseaseOrPhenotypicFeature,Type II (non-insulin-dependent) diabetes mellitus,430,49,479,abstract,Type II (non-insulin-dependent) diabetes mellitus,True
7,10491763,7,3630,GeneOrGeneProduct,insulin,497,7,504,abstract,insulin,True
8,10491763,8,D005947,ChemicalEntity,glucose,518,7,525,abstract,glucose,True
9,10491763,9,3175,GeneOrGeneProduct,HNF-6,620,5,625,abstract,HNF-6,True


# Single-Document Inspection and Validation
Before processing the complete BioRED dataset, I first inspected one training document in detail to understand the dataset structure and confirm that the preprocessing logic was correct.

The document contained two passages: the title and the abstract. Because this BioRED JSON version did not explicitly provide passage-type labels, I assigned the first passage as the title and the second passage as the abstract based on their order. I also preserved the original offset and text of each passage.

Next, I reconstructed the complete document text using the original BioRED passage offsets rather than simply joining the title and abstract with a separator. This ensured that the character positions remained consistent with the gold-standard annotations.

I then extracted the entity annotations from both passages, including the entity ID, biomedical identifier, entity type, entity text, start offset, length, end offset, and passage type. To validate the reconstruction, I used each entity’s offset and length to extract the corresponding text from the reconstructed document.

All entity annotations matched the text obtained from their offsets, confirming that the document reconstruction and entity-parsing approach were correct. After validating the process on this single document, the same logic could be generalised to all documents in the training dataset.

# Processing the Complete Training Dataset
After validating the preprocessing logic on a single BioRED document, I will apply the same process to every document in the training set. For each document, I will assign passage labels, reconstruct the full text using the original passage offsets, extract all entity annotations, and verify whether each annotated entity matches the text at its recorded offset.

The processed document texts and entity annotations will then be stored in structured tables. Any offset mismatches will be recorded separately for inspection. This step will produce the complete processed training dataset required for the later NER and relation extraction stages.

In [12]:
import pandas as pd

def process_document(document):
    passages = document.get("passages", [])

    # Assign passage labels
    for i, passage in enumerate(passages):
        if i == 0:
            passage["passage_type"] = "title"
        elif i == 1:
            passage["passage_type"] = "abstract"
        else:
            passage["passage_type"] = f"passage_{i}"

    # Reconstruct full text using original offsets
    if passages:
        document_length = max(
            passage.get("offset", 0) + len(passage.get("text", ""))
            for passage in passages
        )
    else:
        document_length = 0

    full_text_chars = [" "] * document_length

    for passage in passages:
        passage_text = passage.get("text", "")
        passage_offset = passage.get("offset", 0)

        start = passage_offset
        end = start + len(passage_text)

        full_text_chars[start:end] = list(passage_text)

    full_text = "".join(full_text_chars)

    # Extract and validate entities
    entity_rows = []

    for passage in passages:
        for annotation in passage.get("annotations", []):
            locations = annotation.get("locations", [])

            if not locations:
                continue

            location = locations[0]

            start = location.get("offset")
            length = location.get("length")
            end = start + length

            annotated_text = annotation.get("text", "")
            text_from_offset = full_text[start:end]

            entity_rows.append({
                "document_id": document.get("id"),
                "entity_id": annotation.get("id"),
                "identifier": annotation.get("infons", {}).get("identifier"),
                "entity_type": annotation.get("infons", {}).get("type"),
                "entity_text": annotated_text,
                "offset": start,
                "length": length,
                "end_offset": end,
                "passage_type": passage.get("passage_type"),
                "text_from_offset": text_from_offset,
                "offset_match": annotated_text == text_from_offset
            })

    return {
        "document_id": document.get("id"),
        "full_text": full_text,
        "entities": entity_rows
    }

In [13]:
processed_train_documents = []
all_train_entities = []

for document in train_data["documents"]:
    processed_document = process_document(document)

    processed_train_documents.append({
        "document_id": processed_document["document_id"],
        "full_text": processed_document["full_text"]
    })

    all_train_entities.extend(processed_document["entities"])

In [14]:
train_documents_df = pd.DataFrame(processed_train_documents)
train_entities_df = pd.DataFrame(all_train_entities)

print("Processed training documents:", len(train_documents_df))
print("Extracted training entities:", len(train_entities_df))

display(train_documents_df.head())
display(train_entities_df.head())

Processed training documents: 400
Extracted training entities: 13351


,document_id,full_text
0,10491763,Hepatocyte nuclear factor-6: associations betw...
1,10661407,"Langerin, a novel C-type lectin specific to La..."
2,10788334,Founder mutations in the BRCA1 gene in Polish ...
3,11009181,Apomorphine: an underutilized therapy for Park...
4,11054569,"Rab6c, a new member of the rab gene family, is..."


,document_id,entity_id,identifier,entity_type,entity_text,offset,length,end_offset,passage_type,text_from_offset,offset_match
0,10491763,0,3175,GeneOrGeneProduct,Hepatocyte nuclear factor-6,0,27,27,title,Hepatocyte nuclear factor-6,True
1,10491763,1,D003924,DiseaseOrPhenotypicFeature,type II diabetes,74,16,90,title,type II diabetes,True
2,10491763,2,3630,GeneOrGeneProduct,insulin,140,7,147,title,insulin,True
3,10491763,3,3175,GeneOrGeneProduct,hepatocyte nuclear factor (HNF)-6,184,33,217,abstract,hepatocyte nuclear factor (HNF)-6,True
4,10491763,4,D003924,DiseaseOrPhenotypicFeature,maturity-onset diabetes,292,23,315,abstract,maturity-onset diabetes,True


In [15]:
print(train_entities_df["offset_match"].value_counts())

failed_matches = train_entities_df[
    train_entities_df["offset_match"] == False
]

print("Failed offset matches:", len(failed_matches))

if not failed_matches.empty:
    display(failed_matches.head())


offset_match
True    13351
Name: count, dtype: int64
Failed offset matches: 0


In [16]:
train_documents_df.to_csv(
    "/content/train_documents_processed.csv",
    index=False
)

train_entities_df.to_csv(
    "/content/train_entities_processed.csv",
    index=False
)

print("Processed training data saved successfully.")

Processed training data saved successfully.


## Inspecting Gold-Standard Relations for just the First Document


In [17]:
# Select one sample training document and inspect its raw relations

first_document = train_data["documents"][0]
sample_relations = first_document.get("relations", [])

print("Document ID:", first_document.get("id"))
print("Number of relations:", len(sample_relations))

for relation in sample_relations:
    print(relation)

Document ID: 10491763
Number of relations: 3
{'id': 'R0', 'infons': {'entity1': '3175', 'entity2': 'D003924', 'type': 'Association', 'novel': 'No'}}
{'id': 'R1', 'infons': {'entity1': 'D005947', 'entity2': '3630', 'type': 'Positive_Correlation', 'novel': 'No'}}
{'id': 'R2', 'infons': {'entity1': 'D005947', 'entity2': 'D003924', 'type': 'Association', 'novel': 'No'}}


In [18]:
# Create a lookup using biomedical identifiers

identifier_lookup = {}

for passage in first_document.get("passages", []):
    for annotation in passage.get("annotations", []):
        annotation_infons = annotation.get("infons", {})

        biomedical_identifier = annotation_infons.get("identifier")

        entity_information = {
            "annotation_id": annotation.get("id"),
            "text": annotation.get("text", ""),
            "entity_type": annotation_infons.get("type", ""),
            "identifier": biomedical_identifier
        }

        if biomedical_identifier:
            identifier_lookup[biomedical_identifier] = entity_information

print("Number of identifiers in lookup:", len(identifier_lookup))

Number of identifiers in lookup: 10


In [19]:
# Print readable relation information for the sample document

for relation in sample_relations:
    relation_id = relation.get("id", "")
    relation_infons = relation.get("infons", {})

    relation_type = relation_infons.get("type", "")
    novelty = relation_infons.get("novel", "")

    entity1_id = relation_infons.get("entity1")
    entity2_id = relation_infons.get("entity2")

    entity1 = identifier_lookup.get(entity1_id, {})
    entity2 = identifier_lookup.get(entity2_id, {})

    print("\nRelation ID:", relation_id)
    print("Relation type:", relation_type)
    print("Novelty:", novelty)

    print(
        "Entity 1:",
        entity1_id,
        "|",
        entity1.get("text", "Not found"),
        "|",
        entity1.get("entity_type", "Not found")
    )

    print(
        "Entity 2:",
        entity2_id,
        "|",
        entity2.get("text", "Not found"),
        "|",
        entity2.get("entity_type", "Not found")
    )


Relation ID: R0
Relation type: Association
Novelty: No
Entity 1: 3175 | HNF-6 | GeneOrGeneProduct
Entity 2: D003924 | Type II diabetes | DiseaseOrPhenotypicFeature

Relation ID: R1
Relation type: Positive_Correlation
Novelty: No
Entity 1: D005947 | glucose | ChemicalEntity
Entity 2: 3630 | insulin | GeneOrGeneProduct

Relation ID: R2
Relation type: Association
Novelty: No
Entity 1: D005947 | glucose | ChemicalEntity
Entity 2: D003924 | Type II diabetes | DiseaseOrPhenotypicFeature


## Gold-Standard Relation Extraction

After inspecting the relation structure in one sample BioRED document, the next step is to extract all gold-standard relations from the training split.

In BioRED, each relation stores:

- the first entity identifier
- the second entity identifier
- the relation type
- the novelty label

The biomedical identifiers are matched with the entity annotations in the same document so that each relation can also include the entity text and entity category.

In [20]:
import re
import pandas as pd

all_train_relations = []

for document in train_data["documents"]:
    document_id = document.get("id")

    identifier_lookup = {}

    for passage in document.get("passages", []):
        for annotation in passage.get("annotations", []):
            annotation_infons = annotation.get("infons", {})

            annotation_id = annotation.get("id")
            raw_identifier = annotation_infons.get("identifier")

            entity_information = {
                "annotation_id": annotation_id,
                "text": annotation.get("text", ""),
                "entity_type": annotation_infons.get("type", ""),
                "identifier": raw_identifier
            }

            # Store annotation ID as a fallback
            if annotation_id is not None:
                identifier_lookup[str(annotation_id).strip()] = entity_information

            if raw_identifier is not None:
                raw_identifier = str(raw_identifier).strip()

                # Store the complete identifier value
                identifier_lookup[raw_identifier] = entity_information

                # Store each identifier separately when multiple IDs occur
                individual_identifiers = re.split(
                    r"[,;|]",
                    raw_identifier
                )

                for individual_identifier in individual_identifiers:
                    individual_identifier = individual_identifier.strip()

                    if individual_identifier:
                        identifier_lookup[individual_identifier] = entity_information

    for relation in document.get("relations", []):
        relation_infons = relation.get("infons", {})

        entity1_identifier = relation_infons.get("entity1")
        entity2_identifier = relation_infons.get("entity2")

        if entity1_identifier is not None:
            entity1_identifier = str(entity1_identifier).strip()

        if entity2_identifier is not None:
            entity2_identifier = str(entity2_identifier).strip()

        entity1 = identifier_lookup.get(entity1_identifier, {})
        entity2 = identifier_lookup.get(entity2_identifier, {})

        all_train_relations.append({
            "document_id": document_id,
            "relation_id": relation.get("id"),
            "relation_type": relation_infons.get("type"),
            "novelty": relation_infons.get("novel"),

            "entity1_identifier": entity1_identifier,
            "entity1_text": entity1.get("text"),
            "entity1_type": entity1.get("entity_type"),
            "entity1_annotation_id": entity1.get("annotation_id"),

            "entity2_identifier": entity2_identifier,
            "entity2_text": entity2.get("text"),
            "entity2_type": entity2.get("entity_type"),
            "entity2_annotation_id": entity2.get("annotation_id")
        })


In [21]:
train_relations_df = pd.DataFrame(all_train_relations)

print("Total training relations:", len(train_relations_df))
print("Number of columns:", len(train_relations_df.columns))

display(train_relations_df.head())

Total training relations: 4178
Number of columns: 12


,document_id,relation_id,relation_type,novelty,entity1_identifier,entity1_text,entity1_type,entity1_annotation_id,entity2_identifier,entity2_text,entity2_type,entity2_annotation_id
0,10491763,R0,Association,No,3175,HNF-6,GeneOrGeneProduct,28,D003924,Type II diabetes,DiseaseOrPhenotypicFeature,29
1,10491763,R1,Positive_Correlation,No,D005947,glucose,ChemicalEntity,31,3630,insulin,GeneOrGeneProduct,30
2,10491763,R2,Association,No,D005947,glucose,ChemicalEntity,31,D003924,Type II diabetes,DiseaseOrPhenotypicFeature,29
3,10661407,R0,Bind,Novel,50489,Langerin,GeneOrGeneProduct,9,D008358,mannose,ChemicalEntity,3
4,10788334,R0,Positive_Correlation,Novel,D001943,breast cancer,DiseaseOrPhenotypicFeature,17,c|INS|5382|C,5382insC,SequenceVariant,21


In [22]:
missing_entity1 = train_relations_df["entity1_text"].isna().sum()
missing_entity2 = train_relations_df["entity2_text"].isna().sum()

print("Unresolved Entity 1 references:", missing_entity1)
print("Unresolved Entity 2 references:", missing_entity2)

Unresolved Entity 1 references: 0
Unresolved Entity 2 references: 0


### Note on Repeated Entity Mentions (my future use)

Some biomedical concepts may appear more than once in the same document. At this stage, the lookup keeps one representative mention for each biomedical identifier, which is sufficient for validating the gold-standard relations.

Later, during relation-extraction data preparation, all mentions of the same identifier will be stored and compared. Same-sentence mention pairs will be preferred, and if no same-sentence pair exists, the closest pair will be selected. This will prevent an arbitrary mention from being used in the model input.

### Relation Extraction Summary

A total of 4,178 gold-standard relations were extracted from the BioRED training split. Each relation was mapped to its corresponding biomedical entities using document-level identifier lookups. Validation confirmed that all relation arguments were resolved successfully, with 0 unresolved Entity 1 references and 0 unresolved Entity 2 references.

In [23]:
train_relations_df.to_csv(
    "train_relations_processed.csv",
    index=False
)

print("Saved: train_relations_processed.csv")

Saved: train_relations_processed.csv


##validating offsets for dev and test splits using the earlier function

In [24]:
def process_split(split_data, split_name):
    split_document_records = []
    split_entity_records = []

    for document in split_data["documents"]:
        document_record = process_document(document)

        split_document_records.append(document_record)
        split_entity_records.extend(document_record["entities"])

    split_documents_df = pd.DataFrame(split_document_records)
    split_entities_df = pd.DataFrame(split_entity_records)

    print(f"{split_name} documents:", len(split_documents_df))
    print(f"{split_name} entities:", len(split_entities_df))

    print(f"\n{split_name} offset match counts:")
    print(split_entities_df["offset_match"].value_counts())

    failed_matches = split_entities_df[
        split_entities_df["offset_match"] == False
    ]

    print(f"{split_name} failed offset matches:", len(failed_matches))

    if not failed_matches.empty:
        display(failed_matches.head())

    return split_documents_df, split_entities_df, failed_matches

In [25]:
dev_documents_df, dev_entities_df, dev_failed_matches = process_split(
    dev_data,
    "Development"
)

test_documents_df, test_entities_df, test_failed_matches = process_split(
    test_data,
    "Test"
)

Development documents: 100
Development entities: 3533

Development offset match counts:
offset_match
True    3533
Name: count, dtype: int64
Development failed offset matches: 0
Test documents: 100
Test entities: 3535

Test offset match counts:
offset_match
True    3535
Name: count, dtype: int64
Test failed offset matches: 0


## Stage 1 Summary

BioRED preprocessing and validation were completed for all dataset splits.

- Training split: 400 documents, 13,351 entities, 0 offset mismatches
- Development split: 100 documents, 3,533 entities, 0 offset mismatches
- Test split: 100 documents, 3,535 entities, 0 offset mismatches
- Training gold relations: 4,178
- Unresolved relation references: 0

All entity offsets and relation references were successfully validated before proceeding to model-ready data preparation.

# Stage 2: Model-Ready Data Preparation

After completing and validating the BioRED preprocessing pipeline on the training split, the next stage will prepare the processed data in a format suitable for model training.

For Named Entity Recognition, the document text will be tokenized and the gold-standard entity annotations will be aligned with the corresponding tokens. BIO labels will then be assigned to represent the beginning, inside, and outside of each biomedical entity.

For relation extraction, entity pairs will be generated from each document and matched against the gold-standard relation annotations. Annotated pairs will receive their corresponding relation labels, while unannotated pairs will be assigned a `No_Relation` label.

This stage will produce model-ready NER and relation-extraction examples for the first fine-tuned transformer experiment.

## 2.1 NER Data Preparation

The validated BioRED documents are tokenized with the PubMedBERT tokenizer.
Gold entity spans are aligned to tokens and converted into BIO labels for
model training.

In [26]:
from transformers import AutoTokenizer

model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

In [27]:
sample_document = train_data["documents"][0]

document_id = sample_document["id"]

print("Document ID:", document_id)

Document ID: 10491763


In [28]:
passages = sorted(
    sample_document["passages"],
    key=lambda passage: passage.get("offset", 0)
)

document_length = max(
    passage.get("offset", 0) + len(passage.get("text", ""))
    for passage in passages
)

document_characters = [" "] * document_length

for passage in passages:
    passage_text = passage.get("text", "")
    passage_offset = passage.get("offset", 0)

    document_characters[
        passage_offset:passage_offset + len(passage_text)
    ] = passage_text

document_text = "".join(document_characters)

print(document_text)

Hepatocyte nuclear factor-6: associations between genetic variability and type II diabetes and between genetic variability and estimates of insulin secretion. The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mutations in the MODY1, MODY3 and MODY4 genes and in 54 patients with late-onset Type II diabetes by combined single strand conformational polymorphism-heteroduplex analysis followed by direct sequencing of identified variants. An identified missense variant was examined in association studies and gen

In [29]:
tokenized_document = tokenizer(
    document_text,
    return_offsets_mapping=True,
    truncation=False,
    add_special_tokens=True
)

tokens = tokenizer.convert_ids_to_tokens(
    tokenized_document["input_ids"]
)

offset_mapping = tokenized_document["offset_mapping"]

print("Number of tokens:", len(tokens))

Number of tokens: 352


In [30]:
for token, offsets in list(zip(tokens, offset_mapping))[:30]:
    print(f"{token:<20} {offsets}")

[CLS]                (0, 0)
hepatocyte           (0, 10)
nuclear              (11, 18)
factor               (19, 25)
-                    (25, 26)
6                    (26, 27)
:                    (27, 28)
associations         (29, 41)
between              (42, 49)
genetic              (50, 57)
variability          (58, 69)
and                  (70, 73)
type                 (74, 78)
ii                   (79, 81)
diabetes             (82, 90)
and                  (91, 94)
between              (95, 102)
genetic              (103, 110)
variability          (111, 122)
and                  (123, 126)
estimates            (127, 136)
of                   (137, 139)
insulin              (140, 147)
secretion            (148, 157)
.                    (157, 158)
the                  (159, 162)
transcription        (163, 176)
factor               (177, 183)
hepatocyte           (184, 194)
nuclear              (195, 202)


In [31]:
sample_entities = []

for passage in sample_document["passages"]:
    passage_type = passage.get("passage_type", "")

    for annotation in passage.get("annotations", []):
        entity_start = annotation["locations"][0]["offset"]
        entity_length = annotation["locations"][0]["length"]
        entity_end = entity_start + entity_length

        sample_entities.append({
            "annotation_id": annotation["id"],
            "text": annotation["text"],
            "type": annotation["infons"]["type"],
            "start": entity_start,
            "end": entity_end,
            "passage_type": passage_type
        })

print("Number of entities:", len(sample_entities))

for entity in sample_entities[:10]:
    print(entity)

Number of entities: 32
{'annotation_id': '0', 'text': 'Hepatocyte nuclear factor-6', 'type': 'GeneOrGeneProduct', 'start': 0, 'end': 27, 'passage_type': 'title'}
{'annotation_id': '1', 'text': 'type II diabetes', 'type': 'DiseaseOrPhenotypicFeature', 'start': 74, 'end': 90, 'passage_type': 'title'}
{'annotation_id': '2', 'text': 'insulin', 'type': 'GeneOrGeneProduct', 'start': 140, 'end': 147, 'passage_type': 'title'}
{'annotation_id': '3', 'text': 'hepatocyte nuclear factor (HNF)-6', 'type': 'GeneOrGeneProduct', 'start': 184, 'end': 217, 'passage_type': 'abstract'}
{'annotation_id': '4', 'text': 'maturity-onset diabetes', 'type': 'DiseaseOrPhenotypicFeature', 'start': 292, 'end': 315, 'passage_type': 'abstract'}
{'annotation_id': '5', 'text': 'HNF-6', 'type': 'GeneOrGeneProduct', 'start': 389, 'end': 394, 'passage_type': 'abstract'}
{'annotation_id': '6', 'text': 'Type II (non-insulin-dependent) diabetes mellitus', 'type': 'DiseaseOrPhenotypicFeature', 'start': 430, 'end': 479, 'passa

In [32]:
label_list = ["O"]

for entity in sample_entities:
    entity_type = entity["type"]

    b_label = "B-" + entity_type
    i_label = "I-" + entity_type

    if b_label not in label_list:
        label_list.append(b_label)

    if i_label not in label_list:
        label_list.append(i_label)

label_to_id = {label: index for index, label in enumerate(label_list)}
id_to_label = {index: label for label, index in label_to_id.items()}

print(label_list)
print(label_to_id)

['O', 'B-GeneOrGeneProduct', 'I-GeneOrGeneProduct', 'B-DiseaseOrPhenotypicFeature', 'I-DiseaseOrPhenotypicFeature', 'B-ChemicalEntity', 'I-ChemicalEntity', 'B-OrganismTaxon', 'I-OrganismTaxon', 'B-SequenceVariant', 'I-SequenceVariant']
{'O': 0, 'B-GeneOrGeneProduct': 1, 'I-GeneOrGeneProduct': 2, 'B-DiseaseOrPhenotypicFeature': 3, 'I-DiseaseOrPhenotypicFeature': 4, 'B-ChemicalEntity': 5, 'I-ChemicalEntity': 6, 'B-OrganismTaxon': 7, 'I-OrganismTaxon': 8, 'B-SequenceVariant': 9, 'I-SequenceVariant': 10}


In [33]:
token_labels = ["O"] * len(tokens)

for entity in sample_entities:
    entity_start = entity["start"]
    entity_end = entity["end"]
    entity_type = entity["type"]

    first_token = True

    for token_index, (token_start, token_end) in enumerate(offset_mapping):

        # Ignore special tokens such as [CLS] and [SEP]
        if token_start == token_end:
            continue

        # Check whether token overlaps with entity span
        token_overlaps_entity = (
            token_start < entity_end and token_end > entity_start
        )

        if token_overlaps_entity:
            if first_token:
                token_labels[token_index] = "B-" + entity_type
                first_token = False
            else:
                token_labels[token_index] = "I-" + entity_type

In [34]:
for token, offsets, label in list(zip(tokens, offset_mapping, token_labels))[:80]:
    print(f"{token:<20} {str(offsets):<15} {label}")

[CLS]                (0, 0)          O
hepatocyte           (0, 10)         B-GeneOrGeneProduct
nuclear              (11, 18)        I-GeneOrGeneProduct
factor               (19, 25)        I-GeneOrGeneProduct
-                    (25, 26)        I-GeneOrGeneProduct
6                    (26, 27)        I-GeneOrGeneProduct
:                    (27, 28)        O
associations         (29, 41)        O
between              (42, 49)        O
genetic              (50, 57)        O
variability          (58, 69)        O
and                  (70, 73)        O
type                 (74, 78)        B-DiseaseOrPhenotypicFeature
ii                   (79, 81)        I-DiseaseOrPhenotypicFeature
diabetes             (82, 90)        I-DiseaseOrPhenotypicFeature
and                  (91, 94)        O
between              (95, 102)       O
genetic              (103, 110)      O
variability          (111, 122)      O
and                  (123, 126)      O
estimates            (127, 136)      O
of        

In [35]:
aligned_entity_count = 0
unaligned_entities = []

for entity in sample_entities:
    entity_start = entity["start"]
    entity_end = entity["end"]

    aligned_tokens = []

    for token_index, (token_start, token_end) in enumerate(offset_mapping):

        if token_start == token_end:
            continue

        token_overlaps_entity = (
            token_start < entity_end and token_end > entity_start
        )

        if token_overlaps_entity:
            aligned_tokens.append(tokens[token_index])

    if len(aligned_tokens) > 0:
        aligned_entity_count += 1
    else:
        unaligned_entities.append(entity)

print("Total gold entities:", len(sample_entities))
print("Aligned entities:", aligned_entity_count)
print("Unaligned entities:", len(unaligned_entities))

if len(unaligned_entities) > 0:
    print("\nUnaligned entities:")
    for entity in unaligned_entities:
        print(entity)

Total gold entities: 32
Aligned entities: 32
Unaligned entities: 0


In [36]:
def prepare_ner_document(document, tokenizer):
    passages = sorted(
        document["passages"],
        key=lambda passage: passage.get("offset", 0)
    )

    document_length = max(
        passage.get("offset", 0) + len(passage.get("text", ""))
        for passage in passages
    )

    document_characters = [" "] * document_length

    for passage in passages:
        passage_text = passage.get("text", "")
        passage_offset = passage.get("offset", 0)

        document_characters[
            passage_offset:passage_offset + len(passage_text)
        ] = passage_text

    document_text = "".join(document_characters)

    tokenized_document = tokenizer(
        document_text,
        return_offsets_mapping=True,
        truncation=False,
        add_special_tokens=True
    )

    tokens = tokenizer.convert_ids_to_tokens(
        tokenized_document["input_ids"]
    )

    offset_mapping = tokenized_document["offset_mapping"]

    gold_entities = []

    for passage in passages:
        passage_type = passage.get("passage_type", "")

        for annotation in passage.get("annotations", []):
            entity_start = annotation["locations"][0]["offset"]
            entity_length = annotation["locations"][0]["length"]
            entity_end = entity_start + entity_length

            gold_entities.append({
                "annotation_id": annotation["id"],
                "text": annotation["text"],
                "type": annotation["infons"]["type"],
                "start": entity_start,
                "end": entity_end,
                "passage_type": passage_type
            })

    token_labels = ["O"] * len(tokens)

    for entity in gold_entities:
        entity_start = entity["start"]
        entity_end = entity["end"]
        entity_type = entity["type"]

        first_token = True

        for token_index, (token_start, token_end) in enumerate(offset_mapping):

            if token_start == token_end:
                continue

            token_overlaps_entity = (
                token_start < entity_end and token_end > entity_start
            )

            if token_overlaps_entity:
                if first_token:
                    token_labels[token_index] = "B-" + entity_type
                    first_token = False
                else:
                    token_labels[token_index] = "I-" + entity_type

    aligned_entity_count = 0
    unaligned_entities = []

    for entity in gold_entities:
        entity_start = entity["start"]
        entity_end = entity["end"]

        aligned_tokens = []

        for token_index, (token_start, token_end) in enumerate(offset_mapping):

            if token_start == token_end:
                continue

            token_overlaps_entity = (
                token_start < entity_end and token_end > entity_start
            )

            if token_overlaps_entity:
                aligned_tokens.append(tokens[token_index])

        if len(aligned_tokens) > 0:
            aligned_entity_count += 1
        else:
            unaligned_entities.append(entity)

    return {
        "document_id": document["id"],
        "text": document_text,
        "tokens": tokens,
        "input_ids": tokenized_document["input_ids"],
        "attention_mask": tokenized_document["attention_mask"],
        "offset_mapping": offset_mapping,
        "labels": token_labels,
        "gold_entities": gold_entities,
        "total_entities": len(gold_entities),
        "aligned_entities": aligned_entity_count,
        "unaligned_entities": unaligned_entities
    }

This function combines everything I previously tested for a single document. It can be used to:

reconstruct the document,
tokenize it,
extract entities,
align entities with tokens,
assign BIO labels,
validate alignment.

It returns information including:

document_id
text
tokens
input_ids
attention_mask
offset_mapping
labels
gold_entities
alignment statistics

In [37]:
prepared_sample = prepare_ner_document(sample_document, tokenizer)

print("Document ID:", prepared_sample["document_id"])
print("Total entities:", prepared_sample["total_entities"])
print("Aligned entities:", prepared_sample["aligned_entities"])
print("Unaligned entities:", len(prepared_sample["unaligned_entities"]))

Document ID: 10491763
Total entities: 32
Aligned entities: 32
Unaligned entities: 0


In [38]:
prepared_train_ner = []

total_documents = 0
total_entities = 0
total_aligned_entities = 0
total_unaligned_entities = 0

for document in train_data["documents"]:
    prepared_document = prepare_ner_document(document, tokenizer)

    prepared_train_ner.append(prepared_document)

    total_documents += 1
    total_entities += prepared_document["total_entities"]
    total_aligned_entities += prepared_document["aligned_entities"]
    total_unaligned_entities += len(prepared_document["unaligned_entities"])

print("Prepared training documents:", total_documents)
print("Total gold entities:", total_entities)
print("Total aligned entities:", total_aligned_entities)
print("Total unaligned entities:", total_unaligned_entities)

Prepared training documents: 400
Total gold entities: 13351
Total aligned entities: 13351
Total unaligned entities: 0


In [39]:
def prepare_ner_split(split_data, tokenizer, split_name):
    prepared_split_ner = []

    total_documents = 0
    total_entities = 0
    total_aligned_entities = 0
    total_unaligned_entities = 0

    for document in split_data["documents"]:
        prepared_document = prepare_ner_document(document, tokenizer)

        prepared_split_ner.append(prepared_document)

        total_documents += 1
        total_entities += prepared_document["total_entities"]
        total_aligned_entities += prepared_document["aligned_entities"]
        total_unaligned_entities += len(prepared_document["unaligned_entities"])

    print(f"{split_name} documents:", total_documents)
    print(f"{split_name} gold entities:", total_entities)
    print(f"{split_name} aligned entities:", total_aligned_entities)
    print(f"{split_name} unaligned entities:", total_unaligned_entities)

    return prepared_split_ner

In [40]:
prepared_dev_ner = prepare_ner_split(
    dev_data,
    tokenizer,
    "Development"
)

prepared_test_ner = prepare_ner_split(
    test_data,
    tokenizer,
    "Test"
)

Development documents: 100
Development gold entities: 3533
Development aligned entities: 3533
Development unaligned entities: 0
Test documents: 100
Test gold entities: 3535
Test aligned entities: 3535
Test unaligned entities: 0


In [41]:
all_ner_labels = set()

for prepared_split in [prepared_train_ner, prepared_dev_ner, prepared_test_ner]:
    for document in prepared_split:
        all_ner_labels.update(document["labels"])

all_ner_labels = sorted(list(all_ner_labels))

# Keep O first for readability
all_ner_labels.remove("O")
all_ner_labels = ["O"] + all_ner_labels

label_to_id = {
    label: index
    for index, label in enumerate(all_ner_labels)
}

id_to_label = {
    index: label
    for label, index in label_to_id.items()
}

print("Number of NER labels:", len(all_ner_labels))
print(label_to_id)

Number of NER labels: 13
{'O': 0, 'B-CellLine': 1, 'B-ChemicalEntity': 2, 'B-DiseaseOrPhenotypicFeature': 3, 'B-GeneOrGeneProduct': 4, 'B-OrganismTaxon': 5, 'B-SequenceVariant': 6, 'I-CellLine': 7, 'I-ChemicalEntity': 8, 'I-DiseaseOrPhenotypicFeature': 9, 'I-GeneOrGeneProduct': 10, 'I-OrganismTaxon': 11, 'I-SequenceVariant': 12}


In [42]:
def add_label_ids(prepared_split, label_to_id):
    for document in prepared_split:
        label_ids = []

        for label, offsets in zip(document["labels"], document["offset_mapping"]):
            token_start, token_end = offsets

            # Ignore special tokens such as [CLS] and [SEP]
            if token_start == token_end:
                label_ids.append(-100)
            else:
                label_ids.append(label_to_id[label])

        document["label_ids"] = label_ids

    return prepared_split

In [43]:
prepared_train_ner = add_label_ids(prepared_train_ner, label_to_id)
prepared_dev_ner = add_label_ids(prepared_dev_ner, label_to_id)
prepared_test_ner = add_label_ids(prepared_test_ner, label_to_id)

In [44]:
print(prepared_train_ner[0]["tokens"][:20])
print(prepared_train_ner[0]["labels"][:20])
print(prepared_train_ner[0]["label_ids"][:20])

['[CLS]', 'hepatocyte', 'nuclear', 'factor', '-', '6', ':', 'associations', 'between', 'genetic', 'variability', 'and', 'type', 'ii', 'diabetes', 'and', 'between', 'genetic', 'variability', 'and']
['O', 'B-GeneOrGeneProduct', 'I-GeneOrGeneProduct', 'I-GeneOrGeneProduct', 'I-GeneOrGeneProduct', 'I-GeneOrGeneProduct', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DiseaseOrPhenotypicFeature', 'I-DiseaseOrPhenotypicFeature', 'I-DiseaseOrPhenotypicFeature', 'O', 'O', 'O', 'O', 'O']
[-100, 4, 10, 10, 10, 10, 0, 0, 0, 0, 0, 0, 3, 9, 9, 0, 0, 0, 0, 0]


In [45]:
print("NER labels:")
for label, label_id in label_to_id.items():
    print(label_id, label)

NER labels:
0 O
1 B-CellLine
2 B-ChemicalEntity
3 B-DiseaseOrPhenotypicFeature
4 B-GeneOrGeneProduct
5 B-OrganismTaxon
6 B-SequenceVariant
7 I-CellLine
8 I-ChemicalEntity
9 I-DiseaseOrPhenotypicFeature
10 I-GeneOrGeneProduct
11 I-OrganismTaxon
12 I-SequenceVariant


In [46]:
import json

def save_json(data, file_path):
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)

save_json(prepared_train_ner, "prepared_train_ner.json")
save_json(prepared_dev_ner, "prepared_dev_ner.json")
save_json(prepared_test_ner, "prepared_test_ner.json")

save_json(label_to_id, "ner_label_to_id.json")
save_json(id_to_label, "ner_id_to_label.json")

##Stage 2.1 Summary

In this stage, I converted the BioRED entity annotations into a format that can be used for NER model training. This involved tokenizing the documents, matching the entity spans with the correct tokens, and assigning BIO labels to those tokens. I then prepared the training, development, and test splits in the same way and validated the alignment results. At the end, I saved the prepared NER datasets and label mappings so I could directly use them later for model training.

### Stage 2.2: Relation Extraction Data Preparation

After preparing the NER data, the next step is to prepare relation-extraction examples. BioRED relation annotations connect pairs of biomedical entities using their identifiers. Therefore, each relation must be converted into a structured example containing the document text, the two participating entities, their entity types, and the gold relation label. This stage begins with one training document so that the relation structure and entity mapping can be inspected before applying the process to the full dataset.

In [47]:
sample_relations = sample_document.get("relations", [])

print("Document ID:", sample_document["id"])
print("Number of relations:", len(sample_relations))

for relation in sample_relations[:5]:
    print(relation)

Document ID: 10491763
Number of relations: 3
{'id': 'R0', 'infons': {'entity1': '3175', 'entity2': 'D003924', 'type': 'Association', 'novel': 'No'}}
{'id': 'R1', 'infons': {'entity1': 'D005947', 'entity2': '3630', 'type': 'Positive_Correlation', 'novel': 'No'}}
{'id': 'R2', 'infons': {'entity1': 'D005947', 'entity2': 'D003924', 'type': 'Association', 'novel': 'No'}}


In [48]:
def reconstruct_document_text(document):
    passages = sorted(
        document["passages"],
        key=lambda passage: passage.get("offset", 0)
    )

    document_length = max(
        passage.get("offset", 0) + len(passage.get("text", ""))
        for passage in passages
    )

    document_characters = [" "] * document_length

    for passage in passages:
        passage_text = passage.get("text", "")
        passage_offset = passage.get("offset", 0)

        document_characters[
            passage_offset:passage_offset + len(passage_text)
        ] = passage_text

    return "".join(document_characters)

In [49]:
sample_text = reconstruct_document_text(sample_document)

print(sample_text[:500])

Hepatocyte nuclear factor-6: associations between genetic variability and type II diabetes and between genetic variability and estimates of insulin secretion. The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of ins


In [50]:
def build_identifier_lookup(document):
    identifier_lookup = {}

    for passage in document["passages"]:
        passage_type = passage.get("passage_type", "")

        for annotation in passage.get("annotations", []):
            entity_start = annotation["locations"][0]["offset"]
            entity_length = annotation["locations"][0]["length"]
            entity_end = entity_start + entity_length

            raw_identifier = annotation["infons"].get("identifier")

            entity_record = {
                "annotation_id": annotation["id"],
                "identifier": raw_identifier,
                "text": annotation["text"],
                "type": annotation["infons"].get("type"),
                "start": entity_start,
                "end": entity_end,
                "passage_type": passage_type
            }

            if raw_identifier is None:
                continue

            # Store the original identifier
            identifiers_to_store = [raw_identifier]

            # Also store individual identifiers if multiple IDs are present
            for individual_identifier in raw_identifier.split(","):
                individual_identifier = individual_identifier.strip()

                if individual_identifier not in identifiers_to_store:
                    identifiers_to_store.append(individual_identifier)

            for identifier in identifiers_to_store:
                if identifier not in identifier_lookup:
                    identifier_lookup[identifier] = []

                identifier_lookup[identifier].append(entity_record)

    return identifier_lookup

In [51]:
sample_identifier_lookup = build_identifier_lookup(sample_document)

print("Number of unique identifiers:", len(sample_identifier_lookup))

for identifier, mentions in list(sample_identifier_lookup.items())[:5]:
    print("\nIdentifier:", identifier)
    print("Number of mentions:", len(mentions))
    print("First mention:", mentions[0])

Number of unique identifiers: 10

Identifier: 3175
Number of mentions: 5
First mention: {'annotation_id': '0', 'identifier': '3175', 'text': 'Hepatocyte nuclear factor-6', 'type': 'GeneOrGeneProduct', 'start': 0, 'end': 27, 'passage_type': 'title'}

Identifier: D003924
Number of mentions: 7
First mention: {'annotation_id': '1', 'identifier': 'D003924', 'text': 'type II diabetes', 'type': 'DiseaseOrPhenotypicFeature', 'start': 74, 'end': 90, 'passage_type': 'title'}

Identifier: 3630
Number of mentions: 5
First mention: {'annotation_id': '2', 'identifier': '3630', 'text': 'insulin', 'type': 'GeneOrGeneProduct', 'start': 140, 'end': 147, 'passage_type': 'title'}

Identifier: D005947
Number of mentions: 6
First mention: {'annotation_id': '8', 'identifier': 'D005947', 'text': 'glucose', 'type': 'ChemicalEntity', 'start': 518, 'end': 525, 'passage_type': 'abstract'}

Identifier: 3172,3651,6927
Number of mentions: 1
First mention: {'annotation_id': '10', 'identifier': '3172,3651,6927', 'text

In [52]:
def create_positive_relation_examples(document):
    document_text = reconstruct_document_text(document)
    identifier_lookup = build_identifier_lookup(document)

    relation_examples = []

    for relation in document.get("relations", []):
        relation_type = relation["infons"].get("type")

        entity1_identifier = relation["infons"].get("entity1")
        entity2_identifier = relation["infons"].get("entity2")

        entity1_mentions = identifier_lookup.get(entity1_identifier, [])
        entity2_mentions = identifier_lookup.get(entity2_identifier, [])

        if len(entity1_mentions) == 0 or len(entity2_mentions) == 0:
            continue

        # For now, select the first mention of each biomedical concept.
        # Later, this can be improved using sentence-level or closest-mention selection.
        entity1 = entity1_mentions[0]
        entity2 = entity2_mentions[0]

        relation_examples.append({
            "document_id": document["id"],
            "text": document_text,

            "entity1_identifier": entity1_identifier,
            "entity1_annotation_id": entity1["annotation_id"],
            "entity1_text": entity1["text"],
            "entity1_type": entity1["type"],
            "entity1_start": entity1["start"],
            "entity1_end": entity1["end"],

            "entity2_identifier": entity2_identifier,
            "entity2_annotation_id": entity2["annotation_id"],
            "entity2_text": entity2["text"],
            "entity2_type": entity2["type"],
            "entity2_start": entity2["start"],
            "entity2_end": entity2["end"],

            "relation_label": relation_type
        })

    return relation_examples

In [53]:
sample_relation_examples = create_positive_relation_examples(sample_document)

print("Positive relation examples:", len(sample_relation_examples))

for example in sample_relation_examples[:5]:
    print(example)

Positive relation examples: 3
{'document_id': '10491763', 'text': 'Hepatocyte nuclear factor-6: associations between genetic variability and type II diabetes and between genetic variability and estimates of insulin secretion. The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mutations in the MODY1, MODY3 and MODY4 genes and in 54 patients with late-onset Type II diabetes by combined single strand conformational polymorphism-heteroduplex analysis followed by direct sequencing of identified variants. An iden

In [54]:
print("Gold relations in document:", len(sample_document.get("relations", [])))
print("Created positive examples:", len(sample_relation_examples))

Gold relations in document: 3
Created positive examples: 3


In [55]:
sample_relation_examples_df = pd.DataFrame(sample_relation_examples)

sample_relation_examples_df[
    [
        "document_id",
        "entity1_text",
        "entity1_type",
        "entity2_text",
        "entity2_type",
        "relation_label"
    ]
]

,document_id,entity1_text,entity1_type,entity2_text,entity2_type,relation_label
0,10491763,Hepatocyte nuclear factor-6,GeneOrGeneProduct,type II diabetes,DiseaseOrPhenotypicFeature,Association
1,10491763,glucose,ChemicalEntity,insulin,GeneOrGeneProduct,Positive_Correlation
2,10491763,glucose,ChemicalEntity,type II diabetes,DiseaseOrPhenotypicFeature,Association


In [56]:
def prepare_positive_relation_split(split_data, split_name):
    all_relation_examples = []

    total_documents = 0
    total_gold_relations = 0
    total_created_examples = 0

    for document in split_data["documents"]:
        total_documents += 1

        gold_relations = document.get("relations", [])
        relation_examples = create_positive_relation_examples(document)

        total_gold_relations += len(gold_relations)
        total_created_examples += len(relation_examples)

        all_relation_examples.extend(relation_examples)

    print(f"{split_name} documents:", total_documents)
    print(f"{split_name} gold relations:", total_gold_relations)
    print(f"{split_name} created positive examples:", total_created_examples)
    print(f"{split_name} missing relations:", total_gold_relations - total_created_examples)

    return all_relation_examples

In [57]:
prepared_train_re_positive = prepare_positive_relation_split(
    train_data,
    "Training"
)

Training documents: 400
Training gold relations: 4178
Training created positive examples: 4178
Training missing relations: 0


In [58]:
prepared_dev_re_positive = prepare_positive_relation_split(
    dev_data,
    "Development"
)

prepared_test_re_positive = prepare_positive_relation_split(
    test_data,
    "Test"
)

Development documents: 100
Development gold relations: 1162
Development created positive examples: 1162
Development missing relations: 0
Test documents: 100
Test gold relations: 1163
Test created positive examples: 1163
Test missing relations: 0


Creating a global label mapping for relation labels, similar to what I did for NER.

In [59]:
all_relation_labels = set()

for prepared_split in [
    prepared_train_re_positive,
    prepared_dev_re_positive,
    prepared_test_re_positive
]:
    for example in prepared_split:
        all_relation_labels.add(example["relation_label"])

all_relation_labels = sorted(list(all_relation_labels))

relation_label_to_id = {
    label: index
    for index, label in enumerate(all_relation_labels)
}

relation_id_to_label = {
    index: label
    for label, index in relation_label_to_id.items()
}

print("Number of relation labels:", len(all_relation_labels))

for label, label_id in relation_label_to_id.items():
    print(label_id, label)

Number of relation labels: 8
0 Association
1 Bind
2 Comparison
3 Conversion
4 Cotreatment
5 Drug_Interaction
6 Negative_Correlation
7 Positive_Correlation


In [60]:
def add_relation_label_ids(prepared_split, relation_label_to_id):
    for example in prepared_split:
        example["relation_label_id"] = relation_label_to_id[
            example["relation_label"]
        ]

    return prepared_split

In [61]:
prepared_train_re_positive = add_relation_label_ids(
    prepared_train_re_positive,
    relation_label_to_id
)

prepared_dev_re_positive = add_relation_label_ids(
    prepared_dev_re_positive,
    relation_label_to_id
)

prepared_test_re_positive = add_relation_label_ids(
    prepared_test_re_positive,
    relation_label_to_id
)

In [62]:
print(prepared_train_re_positive[0]["relation_label"])
print(prepared_train_re_positive[0]["relation_label_id"])

Association
0


In [63]:
def add_entity_markers(example):
    text = example["text"]

    entity1_start = example["entity1_start"]
    entity1_end = example["entity1_end"]
    entity2_start = example["entity2_start"]
    entity2_end = example["entity2_end"]

    entity1_type = example["entity1_type"]
    entity2_type = example["entity2_type"]

    entity1_start_marker = f"@{entity1_type}$ "
    entity1_end_marker = f" @/{entity1_type}$"

    entity2_start_marker = f"#{entity2_type}$ "
    entity2_end_marker = f" #/{entity2_type}$"

    spans = [
        {
            "start": entity1_start,
            "end": entity1_end,
            "start_marker": entity1_start_marker,
            "end_marker": entity1_end_marker
        },
        {
            "start": entity2_start,
            "end": entity2_end,
            "start_marker": entity2_start_marker,
            "end_marker": entity2_end_marker
        }
    ]

    # Insert markers from right to left so character offsets do not shift
    spans = sorted(spans, key=lambda span: span["start"], reverse=True)

    marked_text = text

    for span in spans:
        marked_text = (
            marked_text[:span["start"]]
            + span["start_marker"]
            + marked_text[span["start"]:span["end"]]
            + span["end_marker"]
            + marked_text[span["end"]:]
        )

    return marked_text

In [64]:
sample_marked_text = add_entity_markers(prepared_train_re_positive[0])

print(sample_marked_text[:800])

@GeneOrGeneProduct$ Hepatocyte nuclear factor-6 @/GeneOrGeneProduct$: associations between genetic variability and #DiseaseOrPhenotypicFeature$ type II diabetes #/DiseaseOrPhenotypicFeature$ and between genetic variability and estimates of insulin secretion. The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mu


In [65]:
def add_marked_text_to_split(prepared_split):
    for example in prepared_split:
        example["marked_text"] = add_entity_markers(example)

    return prepared_split

In [66]:
prepared_train_re_positive = add_marked_text_to_split(prepared_train_re_positive)
prepared_dev_re_positive = add_marked_text_to_split(prepared_dev_re_positive)
prepared_test_re_positive = add_marked_text_to_split(prepared_test_re_positive)

In [67]:
print(prepared_train_re_positive[0]["marked_text"][:800])
print(prepared_dev_re_positive[0]["marked_text"][:800])
print(prepared_test_re_positive[0]["marked_text"][:800])

@GeneOrGeneProduct$ Hepatocyte nuclear factor-6 @/GeneOrGeneProduct$: associations between genetic variability and #DiseaseOrPhenotypicFeature$ type II diabetes #/DiseaseOrPhenotypicFeature$ and between genetic variability and estimates of insulin secretion. The transcription factor hepatocyte nuclear factor (HNF)-6 is an upstream regulator of several genes involved in the pathogenesis of maturity-onset diabetes of the young. We therefore tested the hypothesis that variability in the HNF-6 gene is associated with subsets of Type II (non-insulin-dependent) diabetes mellitus and estimates of insulin secretion in glucose tolerant subjects.   We cloned the coding region as well as the intron-exon boundaries of the HNF-6 gene. We then examined them on genomic DNA in six MODY probands without mu
#DiseaseOrPhenotypicFeature$ Congenital hypothyroidism #/DiseaseOrPhenotypicFeature$ due to a new deletion in the sodium/iodide symporter protein. OBJECTIVE: Iodide transport defect (ITD) is a rare d

In [68]:
def tokenize_relation_examples(prepared_split, tokenizer, max_length=512):
    for example in prepared_split:
        tokenized = tokenizer(
            example["marked_text"],
            truncation=True,
            padding=False,
            max_length=max_length
        )

        example["input_ids"] = tokenized["input_ids"]
        example["attention_mask"] = tokenized["attention_mask"]

    return prepared_split

In [69]:
prepared_train_re_positive = tokenize_relation_examples(
    prepared_train_re_positive,
    tokenizer
)

prepared_dev_re_positive = tokenize_relation_examples(
    prepared_dev_re_positive,
    tokenizer
)

prepared_test_re_positive = tokenize_relation_examples(
    prepared_test_re_positive,
    tokenizer
)

In [70]:
print("Input IDs length:", len(prepared_train_re_positive[0]["input_ids"]))
print("Attention mask length:", len(prepared_train_re_positive[0]["attention_mask"]))
print("Relation label ID:", prepared_train_re_positive[0]["relation_label_id"])

Input IDs length: 386
Attention mask length: 386
Relation label ID: 0


In [71]:
def save_json(data, file_path):

    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2)

save_json(prepared_train_re_positive, "prepared_train_re_positive.json")
save_json(prepared_dev_re_positive, "prepared_dev_re_positive.json")
save_json(prepared_test_re_positive, "prepared_test_re_positive.json")

save_json(relation_label_to_id, "relation_label_to_id.json")
save_json(relation_id_to_label, "relation_id_to_label.json")

##Stage 2.2 Summary

In this stage, I prepared the BioRED relation data for relation classification. I extracted the positive relation examples from the gold annotations and added special entity markers around the two target entities so the model could clearly identify which pair it needed to classify. I then converted the relation labels into numerical IDs and applied the same preparation process to the training, development, and test splits. Finally, I saved the prepared positive relation datasets and label mappings so they could be reused later during relation-extraction model training.

# Stage 3: Model Fine-Tuning and Evaluation

## Stage 3.1: NER Fine-Tuning

The prepared BioRED NER datasets will now be used to fine-tune a biomedical
transformer model for token classification. The training split will be used
for optimisation, the development split for validation and model selection,
and the test split for final evaluation.

In [72]:
variables_to_check = [
    "prepared_train_ner",
    "prepared_dev_ner",
    "prepared_test_ner",
    "label_to_id",
    "id_to_label",
    "tokenizer"
]

for variable_name in variables_to_check:
    print(
        f"{variable_name}:",
        "available" if variable_name in globals() else "missing"
    )

prepared_train_ner: available
prepared_dev_ner: available
prepared_test_ner: available
label_to_id: available
id_to_label: available
tokenizer: available


In [73]:
print("Training documents:", len(prepared_train_ner))
print("Development documents:", len(prepared_dev_ner))
print("Test documents:", len(prepared_test_ner))
print("Number of NER labels:", len(label_to_id))

Training documents: 400
Development documents: 100
Test documents: 100
Number of NER labels: 13


In [74]:
sample_ner_example = prepared_train_ner[0]

print("Available fields:", sample_ner_example.keys())
print("Document ID:", sample_ner_example["document_id"])

print("Input IDs length:", len(sample_ner_example["input_ids"]))
print("Attention mask length:", len(sample_ner_example["attention_mask"]))
print("Label IDs length:", len(sample_ner_example["label_ids"]))

print("\nFirst 20 tokens:")
print(sample_ner_example["tokens"][:20])

print("\nFirst 20 label IDs:")
print(sample_ner_example["label_ids"][:20])

Available fields: dict_keys(['document_id', 'text', 'tokens', 'input_ids', 'attention_mask', 'offset_mapping', 'labels', 'gold_entities', 'total_entities', 'aligned_entities', 'unaligned_entities', 'label_ids'])
Document ID: 10491763
Input IDs length: 352
Attention mask length: 352
Label IDs length: 352

First 20 tokens:
['[CLS]', 'hepatocyte', 'nuclear', 'factor', '-', '6', ':', 'associations', 'between', 'genetic', 'variability', 'and', 'type', 'ii', 'diabetes', 'and', 'between', 'genetic', 'variability', 'and']

First 20 label IDs:
[-100, 4, 10, 10, 10, 10, 0, 0, 0, 0, 0, 0, 3, 9, 9, 0, 0, 0, 0, 0]


In [75]:
def validate_prepared_ner_split(
    prepared_split,
    split_name,
    number_of_labels
):
    missing_fields_count = 0
    unequal_length_count = 0
    invalid_label_count = 0

    required_fields = {
        "document_id",
        "input_ids",
        "attention_mask",
        "label_ids"
    }

    for example in prepared_split:
        if not required_fields.issubset(example.keys()):
            missing_fields_count += 1
            continue

        input_length = len(example["input_ids"])
        attention_length = len(example["attention_mask"])
        label_length = len(example["label_ids"])

        if not (
            input_length == attention_length == label_length
        ):
            unequal_length_count += 1

        invalid_labels = [
            label_id
            for label_id in example["label_ids"]
            if label_id != -100
            and not 0 <= label_id < number_of_labels
        ]

        if invalid_labels:
            invalid_label_count += 1

    print(f"\n{split_name} split")
    print("Total examples:", len(prepared_split))
    print("Examples with missing fields:", missing_fields_count)
    print("Examples with unequal lengths:", unequal_length_count)
    print("Examples with invalid label IDs:", invalid_label_count)

In [76]:
validate_prepared_ner_split(
    prepared_train_ner,
    "Training",
    len(label_to_id)
)

validate_prepared_ner_split(
    prepared_dev_ner,
    "Development",
    len(label_to_id)
)

validate_prepared_ner_split(
    prepared_test_ner,
    "Test",
    len(label_to_id)
)


Training split
Total examples: 400
Examples with missing fields: 0
Examples with unequal lengths: 0
Examples with invalid label IDs: 0

Development split
Total examples: 100
Examples with missing fields: 0
Examples with unequal lengths: 0
Examples with invalid label IDs: 0

Test split
Total examples: 100
Examples with missing fields: 0
Examples with unequal lengths: 0
Examples with invalid label IDs: 0


In [77]:
import numpy as np

def analyse_sequence_lengths(prepared_split, split_name):
    sequence_lengths = [
        len(example["input_ids"])
        for example in prepared_split
    ]

    print(f"\n{split_name} split")
    print("Number of documents:", len(sequence_lengths))
    print("Minimum length:", min(sequence_lengths))
    print("Maximum length:", max(sequence_lengths))
    print("Average length:", round(np.mean(sequence_lengths), 2))
    print("Median length:", round(np.median(sequence_lengths), 2))
    print(
        "Documents longer than 512 tokens:",
        sum(length > 512 for length in sequence_lengths)
    )

    return sequence_lengths

In [78]:
train_sequence_lengths = analyse_sequence_lengths(
    prepared_train_ner,
    "Training"
)

dev_sequence_lengths = analyse_sequence_lengths(
    prepared_dev_ner,
    "Development"
)

test_sequence_lengths = analyse_sequence_lengths(
    prepared_test_ner,
    "Test"
)


Training split
Number of documents: 400
Minimum length: 54
Maximum length: 722
Average length: 335.9
Median length: 334.0
Documents longer than 512 tokens: 15

Development split
Number of documents: 100
Minimum length: 146
Maximum length: 615
Average length: 354.54
Median length: 353.0
Documents longer than 512 tokens: 3

Test split
Number of documents: 100
Minimum length: 88
Maximum length: 548
Average length: 346.54
Median length: 349.0
Documents longer than 512 tokens: 7


In [79]:
def inspect_longest_document(prepared_split, split_name):
    longest_example = max(
        prepared_split,
        key=lambda example: len(example["input_ids"])
    )

    print(f"\nLongest {split_name} document")
    print("Document ID:", longest_example["document_id"])
    print("Sequence length:", len(longest_example["input_ids"]))


inspect_longest_document(prepared_train_ner, "training")
inspect_longest_document(prepared_dev_ner, "development")
inspect_longest_document(prepared_test_ner, "test")


Longest training document
Document ID: 17379047
Sequence length: 722

Longest development document
Document ID: 19880293
Sequence length: 615

Longest test document
Document ID: 17562951
Sequence length: 548


In [80]:
import re


def find_sentence_spans(text):
    """
    Return sentence character spans as:
    (sentence_start, sentence_end, sentence_text)

    The title and abstract newline is treated as a sentence boundary.
    """
    sentence_spans = []

    # Match text ending in sentence punctuation, a newline,
    # or the end of the document.
    sentence_pattern = re.compile(
        r".+?(?:[.!?](?=\s|$)|\n|$)",
        flags=re.DOTALL
    )

    for match in sentence_pattern.finditer(text):
        sentence_start = match.start()
        sentence_end = match.end()

        sentence_text = text[sentence_start:sentence_end]

        # Remove surrounding whitespace while correcting offsets.
        leading_spaces = len(sentence_text) - len(sentence_text.lstrip())
        trailing_spaces = len(sentence_text) - len(sentence_text.rstrip())

        sentence_start += leading_spaces

        if trailing_spaces > 0:
            sentence_end -= trailing_spaces

        cleaned_sentence = text[sentence_start:sentence_end]

        if cleaned_sentence.strip():
            sentence_spans.append({
                "start": sentence_start,
                "end": sentence_end,
                "text": cleaned_sentence
            })

    return sentence_spans

In [81]:
def map_sentences_to_token_ranges(example):
    sentence_spans = find_sentence_spans(example["text"])
    offset_mapping = example["offset_mapping"]

    sentence_token_ranges = []

    for sentence_number, sentence in enumerate(sentence_spans):
        token_indices = []

        for token_index, (token_start, token_end) in enumerate(offset_mapping):
            # Ignore special tokens such as CLS and SEP.
            if token_start == token_end:
                continue

            # Include tokens that overlap the sentence span.
            if (
                token_end > sentence["start"]
                and token_start < sentence["end"]
            ):
                token_indices.append(token_index)

        if token_indices:
            sentence_token_ranges.append({
                "sentence_id": sentence_number,
                "sentence_start": sentence["start"],
                "sentence_end": sentence["end"],
                "token_start": min(token_indices),
                "token_end": max(token_indices) + 1,
                "text": sentence["text"]
            })

    return sentence_token_ranges

In [82]:
MAX_LENGTH = 512
MAX_CONTENT_LENGTH = MAX_LENGTH - 2


def create_sentence_based_chunks(example):
    sentence_ranges = map_sentences_to_token_ranges(example)

    chunks = []
    current_sentences = []
    current_token_start = None
    current_token_end = None
    chunk_number = 0

    for sentence in sentence_ranges:
        sentence_token_start = sentence["token_start"]
        sentence_token_end = sentence["token_end"]

        if current_token_start is None:
            proposed_length = (
                sentence_token_end - sentence_token_start
            )
        else:
            proposed_length = (
                sentence_token_end - current_token_start
            )

        if (
            current_sentences
            and proposed_length > MAX_CONTENT_LENGTH
        ):
            chunks.append({
                "document_id": example["document_id"],
                "chunk_id": chunk_number,
                "sentence_ids": [
                    item["sentence_id"]
                    for item in current_sentences
                ],
                "token_start": current_token_start,
                "token_end": current_token_end
            })

            chunk_number += 1
            current_sentences = []
            current_token_start = None
            current_token_end = None

        if current_token_start is None:
            current_token_start = sentence_token_start

        current_sentences.append(sentence)
        current_token_end = sentence_token_end

    if current_sentences:
        chunks.append({
            "document_id": example["document_id"],
            "chunk_id": chunk_number,
            "sentence_ids": [
                item["sentence_id"]
                for item in current_sentences
            ],
            "token_start": current_token_start,
            "token_end": current_token_end
        })

    return chunks

In [83]:
def build_sentence_based_ner_chunks(example):
    chunk_ranges = create_sentence_based_chunks(example)

    chunked_examples = []

    for chunk in chunk_ranges:
        token_start = chunk["token_start"]
        token_end = chunk["token_end"]

        chunk_input_ids = (
            [tokenizer.cls_token_id]
            + example["input_ids"][token_start:token_end]
            + [tokenizer.sep_token_id]
        )

        chunk_attention_mask = (
            [1]
            + example["attention_mask"][token_start:token_end]
            + [1]
        )

        chunk_label_ids = (
            [-100]
            + example["label_ids"][token_start:token_end]
            + [-100]
        )

        chunk_tokens = (
            [tokenizer.cls_token]
            + example["tokens"][token_start:token_end]
            + [tokenizer.sep_token]
        )

        chunk_offset_mapping = (
            [(0, 0)]
            + example["offset_mapping"][token_start:token_end]
            + [(0, 0)]
        )

        chunked_examples.append({
            "document_id": example["document_id"],
            "chunk_id": chunk["chunk_id"],
            "sentence_ids": chunk["sentence_ids"],
            "input_ids": chunk_input_ids,
            "attention_mask": chunk_attention_mask,
            "label_ids": chunk_label_ids,
            "tokens": chunk_tokens,
            "offset_mapping": chunk_offset_mapping,
            "original_token_start": token_start,
            "original_token_end": token_end
        })

    return chunked_examples

In [84]:
def build_sentence_based_ner_split(prepared_split):
    chunked_split = []

    for example in prepared_split:
        example_chunks = build_sentence_based_ner_chunks(example)
        chunked_split.extend(example_chunks)

    return chunked_split

In [85]:
sentence_chunked_train_ner = build_sentence_based_ner_split(
    prepared_train_ner
)

sentence_chunked_dev_ner = build_sentence_based_ner_split(
    prepared_dev_ner
)

sentence_chunked_test_ner = build_sentence_based_ner_split(
    prepared_test_ner
)

In [86]:
print("Training")
print("Original documents:", len(prepared_train_ner))
print("Chunked examples:", len(sentence_chunked_train_ner))

print("\nDevelopment")
print("Original documents:", len(prepared_dev_ner))
print("Chunked examples:", len(sentence_chunked_dev_ner))

print("\nTest")
print("Original documents:", len(prepared_test_ner))
print("Chunked examples:", len(sentence_chunked_test_ner))

Training
Original documents: 400
Chunked examples: 415

Development
Original documents: 100
Chunked examples: 103

Test
Original documents: 100
Chunked examples: 107


In [87]:
def validate_sentence_chunked_split(
    chunked_split,
    split_name,
    max_length=512
):
    over_length_count = 0
    unequal_length_count = 0
    invalid_special_label_count = 0
    invalid_label_count = 0

    for example in chunked_split:
        input_length = len(example["input_ids"])
        attention_length = len(example["attention_mask"])
        label_length = len(example["label_ids"])
        offset_length = len(example["offset_mapping"])

        if input_length > max_length:
            over_length_count += 1

        if not (
            input_length
            == attention_length
            == label_length
            == offset_length
        ):
            unequal_length_count += 1

        if (
            example["label_ids"][0] != -100
            or example["label_ids"][-1] != -100
        ):
            invalid_special_label_count += 1

        invalid_labels = [
            label_id
            for label_id in example["label_ids"]
            if label_id != -100
            and not 0 <= label_id < len(label_to_id)
        ]

        if invalid_labels:
            invalid_label_count += 1

    print(f"\n{split_name} split")
    print("Total chunked examples:", len(chunked_split))
    print("Examples longer than 512:", over_length_count)
    print("Examples with unequal lengths:", unequal_length_count)
    print(
        "Examples with incorrect special-token labels:",
        invalid_special_label_count
    )
    print("Examples with invalid label IDs:", invalid_label_count)
    print(
        "Maximum chunk length:",
        max(len(example["input_ids"]) for example in chunked_split)
    )

In [88]:
validate_sentence_chunked_split(
    sentence_chunked_train_ner,
    "Training"
)

validate_sentence_chunked_split(
    sentence_chunked_dev_ner,
    "Development"
)

validate_sentence_chunked_split(
    sentence_chunked_test_ner,
    "Test"
)


Training split
Total chunked examples: 415
Examples longer than 512: 0
Examples with unequal lengths: 0
Examples with incorrect special-token labels: 0
Examples with invalid label IDs: 0
Maximum chunk length: 509

Development split
Total chunked examples: 103
Examples longer than 512: 0
Examples with unequal lengths: 0
Examples with incorrect special-token labels: 0
Examples with invalid label IDs: 0
Maximum chunk length: 511

Test split
Total chunked examples: 107
Examples longer than 512: 0
Examples with unequal lengths: 0
Examples with incorrect special-token labels: 0
Examples with invalid label IDs: 0
Maximum chunk length: 497


In [89]:
from datasets import Dataset, DatasetDict

In [90]:
def prepare_huggingface_ner_records(chunked_split):
    model_ready_records = []

    for example in chunked_split:
        model_ready_records.append({
            "document_id": str(example["document_id"]),
            "chunk_id": int(example["chunk_id"]),
            "input_ids": example["input_ids"],
            "attention_mask": example["attention_mask"],
            "labels": example["label_ids"]
        })

    return model_ready_records

In [91]:
train_ner_records = prepare_huggingface_ner_records(
    sentence_chunked_train_ner
)

dev_ner_records = prepare_huggingface_ner_records(
    sentence_chunked_dev_ner
)

test_ner_records = prepare_huggingface_ner_records(
    sentence_chunked_test_ner
)

In [92]:
print("Training records:", len(train_ner_records))
print("Development records:", len(dev_ner_records))
print("Test records:", len(test_ner_records))

Training records: 415
Development records: 103
Test records: 107


In [93]:
train_ner_dataset = Dataset.from_list(train_ner_records)
dev_ner_dataset = Dataset.from_list(dev_ner_records)
test_ner_dataset = Dataset.from_list(test_ner_records)

In [94]:
ner_dataset = DatasetDict({
    "train": train_ner_dataset,
    "validation": dev_ner_dataset,
    "test": test_ner_dataset
})

In [95]:
print(ner_dataset)

DatasetDict({
    train: Dataset({
        features: ['document_id', 'chunk_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 415
    })
    validation: Dataset({
        features: ['document_id', 'chunk_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 103
    })
    test: Dataset({
        features: ['document_id', 'chunk_id', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 107
    })
})


In [96]:
sample_hf_example = train_ner_dataset[0]

print("Document ID:", sample_hf_example["document_id"])
print("Chunk ID:", sample_hf_example["chunk_id"])

print("Input IDs length:", len(sample_hf_example["input_ids"]))
print("Attention mask length:", len(sample_hf_example["attention_mask"]))
print("Labels length:", len(sample_hf_example["labels"]))

print("\nFirst 20 labels:")
print(sample_hf_example["labels"][:20])

Document ID: 10491763
Chunk ID: 0
Input IDs length: 352
Attention mask length: 352
Labels length: 352

First 20 labels:
[-100, 4, 10, 10, 10, 10, 0, 0, 0, 0, 0, 0, 3, 9, 9, 0, 0, 0, 0, 0]


In [97]:
def validate_huggingface_ner_dataset(dataset, split_name):
    unequal_length_count = 0
    over_length_count = 0
    invalid_label_count = 0

    for example in dataset:
        input_length = len(example["input_ids"])
        attention_length = len(example["attention_mask"])
        label_length = len(example["labels"])

        if not (
            input_length
            == attention_length
            == label_length
        ):
            unequal_length_count += 1

        if input_length > 512:
            over_length_count += 1

        invalid_labels = [
            label_id
            for label_id in example["labels"]
            if label_id != -100
            and not 0 <= label_id < len(label_to_id)
        ]

        if invalid_labels:
            invalid_label_count += 1

    print(f"\n{split_name} split")
    print("Total examples:", len(dataset))
    print("Examples with unequal lengths:", unequal_length_count)
    print("Examples longer than 512:", over_length_count)
    print("Examples with invalid labels:", invalid_label_count)

In [98]:
validate_huggingface_ner_dataset(
    ner_dataset["train"],
    "Training"
)

validate_huggingface_ner_dataset(
    ner_dataset["validation"],
    "Development"
)

validate_huggingface_ner_dataset(
    ner_dataset["test"],
    "Test"
)


Training split
Total examples: 415
Examples with unequal lengths: 0
Examples longer than 512: 0
Examples with invalid labels: 0

Development split
Total examples: 103
Examples with unequal lengths: 0
Examples longer than 512: 0
Examples with invalid labels: 0

Test split
Total examples: 107
Examples with unequal lengths: 0
Examples longer than 512: 0
Examples with invalid labels: 0


In [99]:
MODEL_CHECKPOINT = (
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
)

print("Model checkpoint:", MODEL_CHECKPOINT)
print("Tokenizer checkpoint:", tokenizer.name_or_path)

Model checkpoint: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Tokenizer checkpoint: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract


In [100]:
label_to_id_clean = {
    str(label_name): int(label_id)
    for label_name, label_id in label_to_id.items()
}

id_to_label_clean = {
    int(label_id): str(label_name)
    for label_id, label_name in id_to_label.items()
}

print("Number of labels:", len(label_to_id_clean))
print("Label-to-ID mapping:", label_to_id_clean)
print("ID-to-label mapping:", id_to_label_clean)

Number of labels: 13
Label-to-ID mapping: {'O': 0, 'B-CellLine': 1, 'B-ChemicalEntity': 2, 'B-DiseaseOrPhenotypicFeature': 3, 'B-GeneOrGeneProduct': 4, 'B-OrganismTaxon': 5, 'B-SequenceVariant': 6, 'I-CellLine': 7, 'I-ChemicalEntity': 8, 'I-DiseaseOrPhenotypicFeature': 9, 'I-GeneOrGeneProduct': 10, 'I-OrganismTaxon': 11, 'I-SequenceVariant': 12}
ID-to-label mapping: {0: 'O', 1: 'B-CellLine', 2: 'B-ChemicalEntity', 3: 'B-DiseaseOrPhenotypicFeature', 4: 'B-GeneOrGeneProduct', 5: 'B-OrganismTaxon', 6: 'B-SequenceVariant', 7: 'I-CellLine', 8: 'I-ChemicalEntity', 9: 'I-DiseaseOrPhenotypicFeature', 10: 'I-GeneOrGeneProduct', 11: 'I-OrganismTaxon', 12: 'I-SequenceVariant'}


In [101]:
assert len(label_to_id_clean) == 13
assert len(id_to_label_clean) == 13

assert set(label_to_id_clean.values()) == set(range(13))
assert set(id_to_label_clean.keys()) == set(range(13))

print("NER label mappings validated successfully.")

NER label mappings validated successfully.


In [102]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

In [103]:
from transformers import AutoModelForTokenClassification

ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_to_id_clean),
    label2id=label_to_id_clean,
    id2label=id_to_label_clean
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

N

In [104]:
print("Model type:", ner_model.config.model_type)
print("Number of NER labels:", ner_model.config.num_labels)
print("Maximum position embeddings:", ner_model.config.max_position_embeddings)

print("\nConfigured label mapping:")
print(ner_model.config.label2id)

Model type: bert
Number of NER labels: 13
Maximum position embeddings: 512

Configured label mapping:
{'O': 0, 'B-CellLine': 1, 'B-ChemicalEntity': 2, 'B-DiseaseOrPhenotypicFeature': 3, 'B-GeneOrGeneProduct': 4, 'B-OrganismTaxon': 5, 'B-SequenceVariant': 6, 'I-CellLine': 7, 'I-ChemicalEntity': 8, 'I-DiseaseOrPhenotypicFeature': 9, 'I-GeneOrGeneProduct': 10, 'I-OrganismTaxon': 11, 'I-SequenceVariant': 12}


In [105]:
total_parameters = sum(
    parameter.numel()
    for parameter in ner_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in ner_model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")

Total parameters: 108,901,645
Trainable parameters: 108,901,645


In [106]:
from transformers import DataCollatorForTokenClassification

ner_data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

In [107]:
sample_features = []

for example in [
    ner_dataset["train"][0],
    ner_dataset["train"][1]
]:
    sample_features.append({
        "input_ids": example["input_ids"],
        "attention_mask": example["attention_mask"],
        "labels": example["labels"]
    })

sample_batch = ner_data_collator(sample_features)

print("Batch input_ids shape:", sample_batch["input_ids"].shape)
print("Batch attention_mask shape:", sample_batch["attention_mask"].shape)
print("Batch labels shape:", sample_batch["labels"].shape)

Batch input_ids shape: torch.Size([2, 352])
Batch attention_mask shape: torch.Size([2, 352])
Batch labels shape: torch.Size([2, 352])


In [108]:
print(
    "Number of -100 labels in batch:",
    (sample_batch["labels"] == -100).sum().item()
)

Number of -100 labels in batch: 164


In [109]:
!pip install -q evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00


In [110]:
import numpy as np
import evaluate

seqeval_metric = evaluate.load("seqeval")

In [111]:
def compute_ner_metrics(eval_predictions):
    predictions, labels = eval_predictions

    predicted_label_ids = np.argmax(
        predictions,
        axis=2
    )

    true_predictions = []
    true_labels = []

    for prediction_sequence, label_sequence in zip(
        predicted_label_ids,
        labels
    ):
        filtered_predictions = []
        filtered_labels = []

        for predicted_id, label_id in zip(
            prediction_sequence,
            label_sequence
        ):
            if label_id == -100:
                continue

            filtered_predictions.append(
                id_to_label_clean[int(predicted_id)]
            )

            filtered_labels.append(
                id_to_label_clean[int(label_id)]
            )

        true_predictions.append(filtered_predictions)
        true_labels.append(filtered_labels)

    results = seqeval_metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [112]:
ner_training_dataset = ner_dataset.remove_columns(
    ["document_id", "chunk_id"]
)

print(ner_training_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 415
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 103
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 107
    })
})


In [113]:
from transformers import TrainingArguments

ner_training_args = TrainingArguments(
    output_dir="./pubmedbert_biored_ner",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=20,

    save_total_limit=2,

    report_to="none"
)

In [114]:
print("Learning rate:", ner_training_args.learning_rate)
print("Training batch size:", ner_training_args.per_device_train_batch_size)
print("Evaluation batch size:", ner_training_args.per_device_eval_batch_size)
print("Number of epochs:", ner_training_args.num_train_epochs)
print("Evaluation strategy:", ner_training_args.eval_strategy)
print("Best model metric:", ner_training_args.metric_for_best_model)

Learning rate: 2e-05
Training batch size: 4
Evaluation batch size: 4
Number of epochs: 3
Evaluation strategy: IntervalStrategy.EPOCH
Best model metric: f1


In [115]:
pilot_train_dataset = ner_training_dataset["train"].select(
    range(50)
)

pilot_validation_dataset = ner_training_dataset["validation"].select(
    range(20)
)

print("Pilot training examples:", len(pilot_train_dataset))
print(
    "Pilot validation examples:",
    len(pilot_validation_dataset)
)

Pilot training examples: 50
Pilot validation examples: 20


In [116]:
from transformers import TrainingArguments

pilot_training_args = TrainingArguments(
    output_dir="./pubmedbert_biored_ner_pilot",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=1,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=5,

    save_total_limit=1,

    report_to="none"
)

In [117]:
from transformers import Trainer

pilot_trainer = Trainer(
    model=ner_model,
    args=pilot_training_args,
    train_dataset=pilot_train_dataset,
    eval_dataset=pilot_validation_dataset,
    data_collator=ner_data_collator,
    compute_metrics=compute_ner_metrics
)

In [118]:
print("Trainer created successfully.")
print("Pilot train size:", len(pilot_trainer.train_dataset))
print("Pilot validation size:", len(pilot_trainer.eval_dataset))

Trainer created successfully.
Pilot train size: 50
Pilot validation size: 20


In [119]:
pilot_training_result = pilot_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.482312,1.227067,0.000000,0.000000,0.000000,0.742963


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [120]:
pilot_eval_results = pilot_trainer.evaluate()

print(pilot_eval_results)

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
1.482312,1.227067,1,0.000000,0.000000,0.000000,0.742963


{'eval_loss': 1.227067232131958, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0, 'eval_accuracy': 0.7429634422516985}


In [121]:
from transformers import AutoModelForTokenClassification

full_ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_to_id_clean),
    label2id=label_to_id_clean,
    id2label=id_to_label_clean
)

print("Fresh PubMedBERT NER model loaded successfully.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

N

Fresh PubMedBERT NER model loaded successfully.


In [122]:
from transformers import Trainer

full_ner_trainer = Trainer(
    model=full_ner_model,
    args=ner_training_args,
    train_dataset=ner_training_dataset["train"],
    eval_dataset=ner_training_dataset["validation"],
    data_collator=ner_data_collator,
    compute_metrics=compute_ner_metrics
)

In [123]:
print("Full NER Trainer created successfully.")
print(
    "Training examples:",
    len(full_ner_trainer.train_dataset)
)
print(
    "Validation examples:",
    len(full_ner_trainer.eval_dataset)
)

Full NER Trainer created successfully.
Training examples: 415
Validation examples: 103


In [124]:
full_ner_training_result = full_ner_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.189340,0.163320,0.790387,0.837815,0.813410,0.955267
2,0.111815,0.123144,0.831437,0.889329,0.859409,0.966330
3,0.070446,0.116767,0.842359,0.889329,0.865207,0.968287


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [125]:
full_validation_results = full_ner_trainer.evaluate()

print(full_validation_results)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.070446,0.116767,3,0.842359,0.889329,0.865207,0.968287


{'eval_loss': 0.1167672723531723, 'eval_precision': 0.8423592493297587, 'eval_recall': 0.8893291819983017, 'eval_f1': 0.8652072146495938, 'eval_accuracy': 0.9682872865490441}


In [126]:
test_ner_results = full_ner_trainer.evaluate(
    eval_dataset=ner_training_dataset["test"],
    metric_key_prefix="test"
)

print(test_ner_results)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.070446,0.113685,3,0.850041,0.885149,0.867239,0.968364


{'test_loss': 0.11368460208177567, 'test_precision': 0.850040749796251, 'test_recall': 0.8851485148514852, 'test_f1': 0.8672394678492239, 'test_accuracy': 0.9683636152551228}


In [127]:
print("Test Loss:", test_ner_results["test_loss"])
print("Test Precision:", test_ner_results["test_precision"])
print("Test Recall:", test_ner_results["test_recall"])
print("Test F1:", test_ner_results["test_f1"])
print("Test Accuracy:", test_ner_results["test_accuracy"])

Test Loss: 0.11368460208177567
Test Precision: 0.850040749796251
Test Recall: 0.8851485148514852
Test F1: 0.8672394678492239
Test Accuracy: 0.9683636152551228


In [128]:
test_predictions_output = full_ner_trainer.predict(
    ner_training_dataset["test"]
)

test_logits = test_predictions_output.predictions
test_gold_labels = test_predictions_output.label_ids

print("Prediction logits shape:", test_logits.shape)
print("Gold labels shape:", test_gold_labels.shape)

test_predicted_ids = np.argmax(
    test_logits,
    axis=2
)

print("Predicted label IDs shape:", test_predicted_ids.shape)

Prediction logits shape: (107, 497, 13)
Gold labels shape: (107, 497)
Predicted label IDs shape: (107, 497)


In [129]:
def extract_entities_from_bio(tokens, labels):
    entities = []

    current_tokens = []
    current_type = None

    for token, label in zip(tokens, labels):
        if label == "O":
            if current_tokens:
                entities.append({
                    "text": tokenizer.convert_tokens_to_string(current_tokens),
                    "type": current_type
                })
                current_tokens = []
                current_type = None

        elif label.startswith("B-"):
            if current_tokens:
                entities.append({
                    "text": tokenizer.convert_tokens_to_string(current_tokens),
                    "type": current_type
                })

            current_tokens = [token]
            current_type = label[2:]

        elif label.startswith("I-"):
            entity_type = label[2:]

            if current_tokens and current_type == entity_type:
                current_tokens.append(token)
            else:
                if current_tokens:
                    entities.append({
                        "text": tokenizer.convert_tokens_to_string(current_tokens),
                        "type": current_type
                    })

                current_tokens = [token]
                current_type = entity_type

    if current_tokens:
        entities.append({
            "text": tokenizer.convert_tokens_to_string(current_tokens),
            "type": current_type
        })

    return entities

In [130]:
example_index = 0

example = ner_dataset["test"][example_index]

tokens = tokenizer.convert_ids_to_tokens(
    example["input_ids"]
)

gold_labels = []
predicted_labels = []
valid_tokens = []

for token, gold_id, predicted_id in zip(
    tokens,
    test_gold_labels[example_index],
    test_predicted_ids[example_index]
):
    if gold_id == -100:
        continue

    valid_tokens.append(token)
    gold_labels.append(
        id_to_label_clean[int(gold_id)]
    )
    predicted_labels.append(
        id_to_label_clean[int(predicted_id)]
    )

gold_entities = extract_entities_from_bio(
    valid_tokens,
    gold_labels
)

predicted_entities = extract_entities_from_bio(
    valid_tokens,
    predicted_labels
)

print("Document ID:", example["document_id"])
print("Chunk ID:", example["chunk_id"])

print("\nGold entities:")
for entity in gold_entities:
    print(
        f"- {entity['text']} "
        f"[{entity['type']}]"
    )

print("\nPredicted entities:")
for entity in predicted_entities:
    print(
        f"- {entity['text']} "
        f"[{entity['type']}]"
    )

Document ID: 15485686
Chunk ID: 0

Gold entities:
- scn5a [GeneOrGeneProduct]
- long qt syndrome [DiseaseOrPhenotypicFeature]
- tachycardia [DiseaseOrPhenotypicFeature]
- bradycardia [DiseaseOrPhenotypicFeature]
- long qt syndrome [DiseaseOrPhenotypicFeature]
- lqts [DiseaseOrPhenotypicFeature]
- patient [OrganismTaxon]
- bradycardia [DiseaseOrPhenotypicFeature]
- atrioventricular block [DiseaseOrPhenotypicFeature]
- ventricular tachycardia [DiseaseOrPhenotypicFeature]
- atrioventricular block [DiseaseOrPhenotypicFeature]
- lidocaine [ChemicalEntity]
- mexiletine [ChemicalEntity]
- ventricular tachycardia [DiseaseOrPhenotypicFeature]
- lqts [DiseaseOrPhenotypicFeature]
- na ( v ) 1. 5 [GeneOrGeneProduct]
- sodium [ChemicalEntity]
- g - - > a substitution at codon 1763 [SequenceVariant]
- valine ( gtg ) to a methionine ( atg ) [SequenceVariant]
- tsa201 [CellLine]
- tetrodotoxin [ChemicalEntity]
- lidocaine [ChemicalEntity]
- v1764m [SequenceVariant]
- i1762a [SequenceVariant]
- na ( v 

In [131]:
from collections import Counter

all_gold_entities = []
all_predicted_entities = []

for example_index in range(len(ner_dataset["test"])):
    example = ner_dataset["test"][example_index]

    tokens = tokenizer.convert_ids_to_tokens(
        example["input_ids"]
    )

    valid_tokens = []
    gold_labels = []
    predicted_labels = []

    for token, gold_id, predicted_id in zip(
        tokens,
        test_gold_labels[example_index],
        test_predicted_ids[example_index]
    ):
        if gold_id == -100:
            continue

        valid_tokens.append(token)

        gold_labels.append(
            id_to_label_clean[int(gold_id)]
        )

        predicted_labels.append(
            id_to_label_clean[int(predicted_id)]
        )

    gold_entities = extract_entities_from_bio(
        valid_tokens,
        gold_labels
    )

    predicted_entities = extract_entities_from_bio(
        valid_tokens,
        predicted_labels
    )

    document_id = example["document_id"]
    chunk_id = example["chunk_id"]

    for entity in gold_entities:
        all_gold_entities.append({
            "document_id": document_id,
            "chunk_id": chunk_id,
            "text": entity["text"],
            "type": entity["type"]
        })

    for entity in predicted_entities:
        all_predicted_entities.append({
            "document_id": document_id,
            "chunk_id": chunk_id,
            "text": entity["text"],
            "type": entity["type"]
        })

print("Total gold entities:", len(all_gold_entities))
print("Total predicted entities:", len(all_predicted_entities))

Total gold entities: 3535
Total predicted entities: 3681


In [164]:
# Save PubMedBERT NER test predictions for KG construction

import os
import pandas as pd

save_directory = "/content/drive/MyDrive/Capstone_Project/Stage_3_1_Outputs"
os.makedirs(save_directory, exist_ok=True)

pubmedbert_ner_predictions_df = pd.DataFrame(all_predicted_entities)

pubmedbert_ner_predictions_df = pubmedbert_ner_predictions_df.rename(
    columns={
        "text": "entity_text",
        "type": "entity_type"
    }
)

print("Total predicted entity mentions:",
      len(pubmedbert_ner_predictions_df))

display(pubmedbert_ner_predictions_df.head(10))

Total predicted entity mentions: 3681


,document_id,chunk_id,entity_text,entity_type
0,15485686,0,scn5a,GeneOrGeneProduct
1,15485686,0,long qt syndrome,DiseaseOrPhenotypicFeature
2,15485686,0,tachycardia /,DiseaseOrPhenotypicFeature
3,15485686,0,bradycardia,DiseaseOrPhenotypicFeature
4,15485686,0,congenital long qt syndrome,DiseaseOrPhenotypicFeature
5,15485686,0,lqts,DiseaseOrPhenotypicFeature
6,15485686,0,disturbances,DiseaseOrPhenotypicFeature
7,15485686,0,patient,OrganismTaxon
8,15485686,0,fetal bradycardia,DiseaseOrPhenotypicFeature
9,15485686,0,atrioventricular block,DiseaseOrPhenotypicFeature


In [165]:
csv_path = os.path.join(
    save_directory,
    "pubmedbert_ner_test_predictions.csv"
)

pkl_path = os.path.join(
    save_directory,
    "pubmedbert_ner_test_predictions.pkl"
)

pubmedbert_ner_predictions_df.to_csv(
    csv_path,
    index=False
)

pubmedbert_ner_predictions_df.to_pickle(
    pkl_path
)

print("Saved CSV:", csv_path)
print("Saved pickle:", pkl_path)

Saved CSV: /content/drive/MyDrive/Capstone_Project/Stage_3_1_Outputs/pubmedbert_ner_test_predictions.csv
Saved pickle: /content/drive/MyDrive/Capstone_Project/Stage_3_1_Outputs/pubmedbert_ner_test_predictions.pkl


In [166]:
print(os.listdir(save_directory))

['pubmedbert_ner_test_predictions.csv', 'pubmedbert_ner_test_predictions.pkl']


In [132]:
def entity_key(entity):
    return (
        entity["document_id"],
        entity["chunk_id"],
        entity["text"],
        entity["type"]
    )


gold_counter = Counter(
    entity_key(entity)
    for entity in all_gold_entities
)

predicted_counter = Counter(
    entity_key(entity)
    for entity in all_predicted_entities
)


correct_entities = gold_counter & predicted_counter

false_positive_entities = (
    predicted_counter - gold_counter
)

false_negative_entities = (
    gold_counter - predicted_counter
)


total_correct = sum(correct_entities.values())
total_false_positive = sum(
    false_positive_entities.values()
)
total_false_negative = sum(
    false_negative_entities.values()
)

print("Exact correct entities:", total_correct)
print("False positives:", total_false_positive)
print("False negatives:", total_false_negative)

Exact correct entities: 3129
False positives: 552
False negatives: 406


In [133]:
false_positive_by_type = Counter()

for entity_key_value, count in false_positive_entities.items():
    entity_type = entity_key_value[3]
    false_positive_by_type[entity_type] += count


false_negative_by_type = Counter()

for entity_key_value, count in false_negative_entities.items():
    entity_type = entity_key_value[3]
    false_negative_by_type[entity_type] += count


print("False positives by entity type:")
for entity_type, count in false_positive_by_type.most_common():
    print(f"{entity_type}: {count}")


print("\nFalse negatives by entity type:")
for entity_type, count in false_negative_by_type.most_common():
    print(f"{entity_type}: {count}")

False positives by entity type:
DiseaseOrPhenotypicFeature: 223
ChemicalEntity: 143
GeneOrGeneProduct: 90
SequenceVariant: 58
OrganismTaxon: 25
CellLine: 13

False negatives by entity type:
DiseaseOrPhenotypicFeature: 154
GeneOrGeneProduct: 116
ChemicalEntity: 60
OrganismTaxon: 36
SequenceVariant: 30
CellLine: 10


In [134]:
final_ner_model_path = "./final_pubmedbert_biored_ner"

full_ner_trainer.save_model(
    final_ner_model_path
)

tokenizer.save_pretrained(
    final_ner_model_path
)

print(
    "Final NER model and tokenizer saved to:",
    final_ner_model_path
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final NER model and tokenizer saved to: ./final_pubmedbert_biored_ner


In [135]:
ner_results_summary = {
    "validation": {
        "precision": float(full_validation_results["eval_precision"]),
        "recall": float(full_validation_results["eval_recall"]),
        "f1": float(full_validation_results["eval_f1"]),
        "accuracy": float(full_validation_results["eval_accuracy"]),
        "loss": float(full_validation_results["eval_loss"])
    },

    "test": {
        "precision": float(test_ner_results["test_precision"]),
        "recall": float(test_ner_results["test_recall"]),
        "f1": float(test_ner_results["test_f1"]),
        "accuracy": float(test_ner_results["test_accuracy"]),
        "loss": float(test_ner_results["test_loss"])
    }
}

with open(
    "./ner_evaluation_results.json",
    "w"
) as results_file:
    json.dump(
        ner_results_summary,
        results_file,
        indent=4
    )

print("NER evaluation results saved successfully.")

NER evaluation results saved successfully.


##Stage 3.1 Summary

In this stage, I fine-tuned PubMedBERT for biomedical Named Entity Recognition using the prepared BioRED datasets. I first validated the training, development, and test data and analysed the sequence lengths to identify documents exceeding the model’s 512-token limit. These longer documents were handled using sentence-based chunking while keeping the BIO labels correctly aligned with the tokens.

The chunked data was then converted into Hugging Face datasets and prepared for token classification. Before running the full training, I carried out a smaller pilot training run to make sure the complete training pipeline was working correctly. After this validation, PubMedBERT was fine-tuned on the full training set and evaluated on both the development and test sets using precision, recall, and F1-score.

Finally, I analysed the model’s predictions to identify false positives, false negatives, and errors across different entity types. The model achieved an F1-score of 86.54% on the BioRED test set, and the final trained model, tokenizer, and evaluation results were saved for later use.

### Stage 3.2: Relation Extraction Data Completion

Now that the NER model has been trained and evaluated, the next step is to complete the dataset needed for relation extraction.

So far, the positive relation examples from BioRED have already been prepared and entity markers have been added successfully. However, for relation classification, the model also needs to learn when two entities do not have a relation.

In this stage, possible entity pairs will be created from each document. Any pair that already has a gold relation will be kept as a positive example, while suitable pairs without a labelled relation will be added as `No_Relation` examples.

Since the number of possible negative pairs can become very large, the negative examples will also need to be controlled so that the dataset does not become too imbalanced.

By the end of this stage, the relation extraction dataset will contain both the original positive relation examples and the new `No_Relation` examples, ready for PubMedBERT fine-tuning and evaluation.


In [136]:
# This step reconstructs each BioRED training document using the original
# passage offsets so that entity character positions stay correctly aligned.

def reconstruct_document_text(document):
    passages = document["passages"]

    document_length = max(
        passage["offset"] + len(passage.get("text", ""))
        for passage in passages
    )

    characters = [" "] * document_length

    for passage in passages:
        start = passage["offset"]
        passage_text = passage.get("text", "")

        characters[start:start + len(passage_text)] = passage_text

    return "".join(characters)


train_document_text_lookup = {
    document["id"]: reconstruct_document_text(document)
    for document in train_data["documents"]
}

print(
    "Number of reconstructed training documents:",
    len(train_document_text_lookup)
)

Number of reconstructed training documents: 400


In [137]:
# This step creates No_Relation candidates while excluding invalid pairs
# where both identifiers refer to the exact same entity mention span.

from itertools import combinations


def generate_no_relation_candidates(data):
    no_relation_candidates = []

    for document in data["documents"]:
        document_id = document["id"]

        # Keep one representative mention for each unique biomedical identifier
        unique_entities = {}

        for passage in document["passages"]:
            for annotation in passage.get("annotations", []):

                identifiers = annotation["infons"].get("identifier", "").split(",")

                for identifier in identifiers:
                    identifier = identifier.strip()

                    if identifier and identifier not in unique_entities:
                        location = annotation["locations"][0]

                        unique_entities[identifier] = {
                            "identifier": identifier,
                            "annotation_id": annotation["id"],
                            "text": annotation["text"],
                            "type": annotation["infons"]["type"],
                            "start": location["offset"],
                            "end": location["offset"] + location["length"]
                        }

        # Store all existing gold relation pairs
        gold_relation_pairs = set()

        for relation in document.get("relations", []):
            entity1_identifier = relation["infons"]["entity1"]
            entity2_identifier = relation["infons"]["entity2"]

            pair = tuple(sorted([
                entity1_identifier,
                entity2_identifier
            ]))

            gold_relation_pairs.add(pair)

        # Generate possible negative pairs
        entity_identifiers = list(unique_entities.keys())

        for entity1_identifier, entity2_identifier in combinations(
            entity_identifiers, 2
        ):

            entity1 = unique_entities[entity1_identifier]
            entity2 = unique_entities[entity2_identifier]

            # Skip pairs that refer to the exact same text span
            if (
                entity1["start"] == entity2["start"]
                and entity1["end"] == entity2["end"]
            ):
                continue

            pair = tuple(sorted([
                entity1_identifier,
                entity2_identifier
            ]))

            # Keep only pairs that do not already have a gold relation
            if pair not in gold_relation_pairs:

                no_relation_candidates.append({
                    "document_id": document_id,

                    "entity1_identifier": entity1["identifier"],
                    "entity1_annotation_id": entity1["annotation_id"],
                    "entity1_text": entity1["text"],
                    "entity1_type": entity1["type"],
                    "entity1_start": entity1["start"],
                    "entity1_end": entity1["end"],

                    "entity2_identifier": entity2["identifier"],
                    "entity2_annotation_id": entity2["annotation_id"],
                    "entity2_text": entity2["text"],
                    "entity2_type": entity2["type"],
                    "entity2_start": entity2["start"],
                    "entity2_end": entity2["end"],

                    "relation_label": "No_Relation"
                })

    return no_relation_candidates
# Generate the corrected training negative candidates
train_no_relation_candidates = generate_no_relation_candidates(train_data)

print(
    "Number of training No_Relation candidates:",
    len(train_no_relation_candidates)
)

Number of training No_Relation candidates: 26592


In [138]:
# This step randomly selects the same number of negative examples
# as positive training relations to keep the training set balanced.

import random

random.seed(42)

num_positive_train = len(prepared_train_re_positive)

sampled_train_no_relation = random.sample(
    train_no_relation_candidates,
    num_positive_train
)

print("Positive training examples:", num_positive_train)
print("Sampled No_Relation examples:", len(sampled_train_no_relation))
print(
    "Total training examples after combining:",
    num_positive_train + len(sampled_train_no_relation)
)

Positive training examples: 4178
Sampled No_Relation examples: 4178
Total training examples after combining: 8356


In [139]:
# This step adds No_Relation as the ninth relation class
# so negative examples can be assigned their own label ID.

if "No_Relation" not in relation_label_to_id:
    relation_label_to_id["No_Relation"] = len(relation_label_to_id)

relation_id_to_label = {
    label_id: label
    for label, label_id in relation_label_to_id.items()
}

print("Number of relation labels:", len(relation_label_to_id))
print("No_Relation label ID:", relation_label_to_id["No_Relation"])
print("\nRelation label mapping:")
print(relation_label_to_id)

Number of relation labels: 9
No_Relation label ID: 8

Relation label mapping:
{'Association': 0, 'Bind': 1, 'Comparison': 2, 'Conversion': 3, 'Cotreatment': 4, 'Drug_Interaction': 5, 'Negative_Correlation': 6, 'Positive_Correlation': 7, 'No_Relation': 8}


In [140]:
# This step defines how entity markers are inserted around the two target
# entities while preserving their original character positions.

def insert_entity_markers(
    text,
    entity1_start,
    entity1_end,
    entity1_type,
    entity2_start,
    entity2_end,
    entity2_type
):
    entities = [
        (
            entity1_start,
            entity1_end,
            f"@{entity1_type}$ ",
            f" @/{entity1_type}$"
        ),
        (
            entity2_start,
            entity2_end,
            f"#{entity2_type}$ ",
            f" #/{entity2_type}$"
        )
    ]

    # Insert markers from right to left so earlier offsets remain unchanged
    entities = sorted(
        entities,
        key=lambda x: x[0],
        reverse=True
    )

    marked_text = text

    for start, end, opening_marker, closing_marker in entities:
        marked_text = (
            marked_text[:start]
            + opening_marker
            + marked_text[start:end]
            + closing_marker
            + marked_text[end:]
        )

    return marked_text

In [141]:
# This step creates one reusable function for preparing No_Relation examples
# so the same logic can be used consistently for train, development, and test data.

def prepare_no_relation_examples(
    sampled_examples,
    document_text_lookup,
    tokenizer,
    relation_label_to_id
):
    prepared_examples = []

    for example in sampled_examples:

        text = document_text_lookup[example["document_id"]]

        marked_text = insert_entity_markers(
            text=text,
            entity1_start=example["entity1_start"],
            entity1_end=example["entity1_end"],
            entity1_type=example["entity1_type"],
            entity2_start=example["entity2_start"],
            entity2_end=example["entity2_end"],
            entity2_type=example["entity2_type"]
        )

        encoded = tokenizer(
            marked_text,
            truncation=True
        )

        prepared_example = example.copy()

        prepared_example["text"] = text
        prepared_example["relation_label"] = "No_Relation"
        prepared_example["relation_label_id"] = relation_label_to_id["No_Relation"]
        prepared_example["marked_text"] = marked_text
        prepared_example["input_ids"] = encoded["input_ids"]
        prepared_example["attention_mask"] = encoded["attention_mask"]

        prepared_examples.append(prepared_example)

    return prepared_examples

In [142]:
# This step prepares the sampled No_Relation examples by adding the
# reconstructed document text, entity markers, label IDs, and tokenized inputs.

prepared_train_re_negative = []

for example in sampled_train_no_relation:

    text = train_document_text_lookup[example["document_id"]]

    marked_text = insert_entity_markers(
        text=text,
        entity1_start=example["entity1_start"],
        entity1_end=example["entity1_end"],
        entity1_type=example["entity1_type"],
        entity2_start=example["entity2_start"],
        entity2_end=example["entity2_end"],
        entity2_type=example["entity2_type"]
    )

    encoded = tokenizer(
        marked_text,
        truncation=True
    )

    prepared_example = example.copy()

    prepared_example["text"] = text
    prepared_example["relation_label"] = "No_Relation"
    prepared_example["relation_label_id"] = relation_label_to_id["No_Relation"]
    prepared_example["marked_text"] = marked_text
    prepared_example["input_ids"] = encoded["input_ids"]
    prepared_example["attention_mask"] = encoded["attention_mask"]

    prepared_train_re_negative.append(prepared_example)


print(
    "Prepared training No_Relation examples:",
    len(prepared_train_re_negative)
)

Prepared training No_Relation examples: 4178


In [143]:
# This step validates the corrected No_Relation examples by checking
# entity markers, relation labels, and tokenized inputs.

missing_entity1_markers = 0
missing_entity2_markers = 0
wrong_labels = 0
empty_input_ids = 0

for example in prepared_train_re_negative:

    marked_text = example["marked_text"]

    if (
        f"@{example['entity1_type']}$" not in marked_text
        or f"@/{example['entity1_type']}$" not in marked_text
    ):
        missing_entity1_markers += 1

    if (
        f"#{example['entity2_type']}$" not in marked_text
        or f"#/{example['entity2_type']}$" not in marked_text
    ):
        missing_entity2_markers += 1

    if example["relation_label"] != "No_Relation":
        wrong_labels += 1

    if len(example["input_ids"]) == 0:
        empty_input_ids += 1


print("Total negative examples:", len(prepared_train_re_negative))
print("Missing entity 1 markers:", missing_entity1_markers)
print("Missing entity 2 markers:", missing_entity2_markers)
print("Incorrect relation labels:", wrong_labels)
print("Empty input IDs:", empty_input_ids)

Total negative examples: 4178
Missing entity 1 markers: 0
Missing entity 2 markers: 0
Incorrect relation labels: 0
Empty input IDs: 0


In [144]:
# This step combines the positive and No_Relation training examples
# into one balanced relation extraction training dataset and shuffles it.

prepared_train_re = (
    prepared_train_re_positive
    + prepared_train_re_negative
)

random.seed(42)
random.shuffle(prepared_train_re)

print("Positive examples:", len(prepared_train_re_positive))
print("Negative examples:", len(prepared_train_re_negative))
print("Total combined training examples:", len(prepared_train_re))

Positive examples: 4178
Negative examples: 4178
Total combined training examples: 8356


In [145]:
# This step generates valid No_Relation candidates for the development split
# using the same corrected candidate-generation function as the training data.

dev_no_relation_candidates = generate_no_relation_candidates(dev_data)

print(
    "Number of development No_Relation candidates:",
    len(dev_no_relation_candidates)
)

Number of development No_Relation candidates: 7866


In [146]:
# This step randomly selects the same number of development negative examples
# as positive development relations to keep the development set balanced.

random.seed(42)

num_positive_dev = len(prepared_dev_re_positive)

sampled_dev_no_relation = random.sample(
    dev_no_relation_candidates,
    num_positive_dev
)

print("Positive development examples:", num_positive_dev)
print("Sampled No_Relation examples:", len(sampled_dev_no_relation))
print(
    "Total development examples after combining:",
    num_positive_dev + len(sampled_dev_no_relation)
)

Positive development examples: 1162
Sampled No_Relation examples: 1162
Total development examples after combining: 2324


In [147]:
# This step reconstructs the development documents using the original
# passage offsets so entity positions remain correctly aligned.

dev_document_text_lookup = {
    document["id"]: reconstruct_document_text(document)
    for document in dev_data["documents"]
}

print(
    "Number of reconstructed development documents:",
    len(dev_document_text_lookup)
)

Number of reconstructed development documents: 100


In [148]:
# This step prepares the sampled development No_Relation examples
# using the reusable preparation function.

prepared_dev_re_negative = prepare_no_relation_examples(
    sampled_examples=sampled_dev_no_relation,
    document_text_lookup=dev_document_text_lookup,
    tokenizer=tokenizer,
    relation_label_to_id=relation_label_to_id
)

print(
    "Prepared development No_Relation examples:",
    len(prepared_dev_re_negative)
)

Prepared development No_Relation examples: 1162


In [149]:
# This step validates the prepared development No_Relation examples
# by checking markers, labels, and tokenized inputs.

missing_entity1_markers = 0
missing_entity2_markers = 0
wrong_labels = 0
empty_input_ids = 0

for example in prepared_dev_re_negative:

    marked_text = example["marked_text"]

    if (
        f"@{example['entity1_type']}$" not in marked_text
        or f"@/{example['entity1_type']}$" not in marked_text
    ):
        missing_entity1_markers += 1

    if (
        f"#{example['entity2_type']}$" not in marked_text
        or f"#/{example['entity2_type']}$" not in marked_text
    ):
        missing_entity2_markers += 1

    if example["relation_label"] != "No_Relation":
        wrong_labels += 1

    if len(example["input_ids"]) == 0:
        empty_input_ids += 1


print("Total development negative examples:", len(prepared_dev_re_negative))
print("Missing entity 1 markers:", missing_entity1_markers)
print("Missing entity 2 markers:", missing_entity2_markers)
print("Incorrect relation labels:", wrong_labels)
print("Empty input IDs:", empty_input_ids)

Total development negative examples: 1162
Missing entity 1 markers: 0
Missing entity 2 markers: 0
Incorrect relation labels: 0
Empty input IDs: 0


In [150]:
# This step combines the positive and No_Relation development examples
# into one balanced development relation extraction dataset.

prepared_dev_re = (
    prepared_dev_re_positive
    + prepared_dev_re_negative
)

print("Positive development examples:", len(prepared_dev_re_positive))
print("Negative development examples:", len(prepared_dev_re_negative))
print("Total combined development examples:", len(prepared_dev_re))

Positive development examples: 1162
Negative development examples: 1162
Total combined development examples: 2324


In [151]:
# This step generates valid No_Relation candidates for the test split
# using the same corrected candidate-generation function.

test_no_relation_candidates = generate_no_relation_candidates(test_data)

print(
    "Number of test No_Relation candidates:",
    len(test_no_relation_candidates)
)

Number of test No_Relation candidates: 8889


In [152]:
# This step randomly selects the same number of test negative examples
# as positive test relations to keep the test set balanced.

random.seed(42)

num_positive_test = len(prepared_test_re_positive)

sampled_test_no_relation = random.sample(
    test_no_relation_candidates,
    num_positive_test
)

print("Positive test examples:", num_positive_test)
print("Sampled No_Relation examples:", len(sampled_test_no_relation))
print(
    "Total test examples after combining:",
    num_positive_test + len(sampled_test_no_relation)
)

Positive test examples: 1163
Sampled No_Relation examples: 1163
Total test examples after combining: 2326


In [153]:
# This step reconstructs the test documents using the original
# passage offsets so entity positions remain correctly aligned.

test_document_text_lookup = {
    document["id"]: reconstruct_document_text(document)
    for document in test_data["documents"]
}

print(
    "Number of reconstructed test documents:",
    len(test_document_text_lookup)
)

Number of reconstructed test documents: 100


In [154]:
# This step prepares the sampled test No_Relation examples
# using the reusable preparation function.

prepared_test_re_negative = prepare_no_relation_examples(
    sampled_examples=sampled_test_no_relation,
    document_text_lookup=test_document_text_lookup,
    tokenizer=tokenizer,
    relation_label_to_id=relation_label_to_id
)

print(
    "Prepared test No_Relation examples:",
    len(prepared_test_re_negative)
)

Prepared test No_Relation examples: 1163


In [155]:
# This step validates the prepared test No_Relation examples
# by checking markers, labels, and tokenized inputs.

missing_entity1_markers = 0
missing_entity2_markers = 0
wrong_labels = 0
empty_input_ids = 0

for example in prepared_test_re_negative:

    marked_text = example["marked_text"]

    if (
        f"@{example['entity1_type']}$" not in marked_text
        or f"@/{example['entity1_type']}$" not in marked_text
    ):
        missing_entity1_markers += 1

    if (
        f"#{example['entity2_type']}$" not in marked_text
        or f"#/{example['entity2_type']}$" not in marked_text
    ):
        missing_entity2_markers += 1

    if example["relation_label"] != "No_Relation":
        wrong_labels += 1

    if len(example["input_ids"]) == 0:
        empty_input_ids += 1


print("Total test negative examples:", len(prepared_test_re_negative))
print("Missing entity 1 markers:", missing_entity1_markers)
print("Missing entity 2 markers:", missing_entity2_markers)
print("Incorrect relation labels:", wrong_labels)
print("Empty input IDs:", empty_input_ids)

Total test negative examples: 1163
Missing entity 1 markers: 0
Missing entity 2 markers: 0
Incorrect relation labels: 0
Empty input IDs: 0


In [156]:
# This step combines the positive and No_Relation test examples
# into one balanced test relation extraction dataset.

prepared_test_re = (
    prepared_test_re_positive
    + prepared_test_re_negative
)

print("Positive test examples:", len(prepared_test_re_positive))
print("Negative test examples:", len(prepared_test_re_negative))
print("Total combined test examples:", len(prepared_test_re))

Positive test examples: 1163
Negative test examples: 1163
Total combined test examples: 2326


##Stage 3.2 Summary

In this stage, I fine-tuned PubMedBERT for the biomedical Named Entity Recognition task using the prepared BioRED datasets. I first checked the input sequence lengths and found that some documents were longer than PubMedBERT’s 512-token limit, so I used chunking to process these longer documents without losing important entity information. The chunked training, development, and test datasets were then prepared for model training while keeping the BIO labels correctly aligned with the tokens.

After that, I fine-tuned PubMedBERT on the training set and used the development set to monitor the model during training. The final model was evaluated on the unseen test set using precision, recall, and F1-score. I also performed error analysis to understand the types of mistakes made by the model. The fine-tuned PubMedBERT model achieved an F1-score of 86.54% on the BioRED test set, and the trained model was saved for later use.

# Stage 3.3: PubMedBERT Relation Extraction Fine-Tuning

In [157]:
# This step checks that the final train, development, and test
# relation extraction datasets and label mappings are ready for fine-tuning.

print("Training examples:", len(prepared_train_re))
print("Development examples:", len(prepared_dev_re))
print("Test examples:", len(prepared_test_re))

print("\nNumber of relation labels:", len(relation_label_to_id))
print("Relation label mapping:", relation_label_to_id)

print("\nSample training example fields:")
print(prepared_train_re[0].keys())

Training examples: 8356
Development examples: 2324
Test examples: 2326

Number of relation labels: 9
Relation label mapping: {'Association': 0, 'Bind': 1, 'Comparison': 2, 'Conversion': 3, 'Cotreatment': 4, 'Drug_Interaction': 5, 'Negative_Correlation': 6, 'Positive_Correlation': 7, 'No_Relation': 8}

Sample training example fields:
dict_keys(['document_id', 'text', 'entity1_identifier', 'entity1_annotation_id', 'entity1_text', 'entity1_type', 'entity1_start', 'entity1_end', 'entity2_identifier', 'entity2_annotation_id', 'entity2_text', 'entity2_type', 'entity2_start', 'entity2_end', 'relation_label', 'relation_label_id', 'marked_text', 'input_ids', 'attention_mask'])


In [158]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [159]:
import os

save_directory = "/content/drive/MyDrive/Capstone_Project/Stage_3_2_Outputs"
os.makedirs(save_directory, exist_ok=True)

print("Save directory:", save_directory)

Save directory: /content/drive/MyDrive/Capstone_Project/Stage_3_2_Outputs


In [160]:
variables_to_check = [
    "prepared_train_re",
    "prepared_dev_re",
    "prepared_test_re",
    "relation_label_to_id",
    "relation_id_to_label"
]

for variable_name in variables_to_check:
    print(variable_name, ":", variable_name in globals())

prepared_train_re : True
prepared_dev_re : True
prepared_test_re : True
relation_label_to_id : True
relation_id_to_label : True


In [161]:
import pickle
import os

re_data = {
    "train": prepared_train_re,
    "development": prepared_dev_re,
    "test": prepared_test_re
}

re_data_path = os.path.join(
    save_directory,
    "biored_prepared_relation_data.pkl"
)

with open(re_data_path, "wb") as file:
    pickle.dump(re_data, file)

print("Relation datasets saved successfully.")
print("Saved to:", re_data_path)

Relation datasets saved successfully.
Saved to: /content/drive/MyDrive/Capstone_Project/Stage_3_2_Outputs/biored_prepared_relation_data.pkl


In [162]:
relation_mappings = {
    "label_to_id": relation_label_to_id,
    "id_to_label": relation_id_to_label
}

mapping_path = os.path.join(
    save_directory,
    "biored_relation_label_mappings.pkl"
)

with open(mapping_path, "wb") as file:
    pickle.dump(relation_mappings, file)

print("Relation mappings saved successfully.")
print("Saved to:", mapping_path)

Relation mappings saved successfully.
Saved to: /content/drive/MyDrive/Capstone_Project/Stage_3_2_Outputs/biored_relation_label_mappings.pkl


In [163]:
with open(re_data_path, "rb") as file:
    loaded_re_data = pickle.load(file)

with open(mapping_path, "rb") as file:
    loaded_relation_mappings = pickle.load(file)

print("Training examples:", len(loaded_re_data["train"]))
print("Development examples:", len(loaded_re_data["development"]))
print("Test examples:", len(loaded_re_data["test"]))

print("\nRelation label mapping:")
print(loaded_relation_mappings["label_to_id"])

Training examples: 8356
Development examples: 2324
Test examples: 2326

Relation label mapping:
{'Association': 0, 'Bind': 1, 'Comparison': 2, 'Conversion': 3, 'Cotreatment': 4, 'Drug_Interaction': 5, 'Negative_Correlation': 6, 'Positive_Correlation': 7, 'No_Relation': 8}
